### [Class 4] Question 1. What Two Target-Module Presets Cost in Trainable Parameters

**Context:** Choosing which weight matrices receive an adapter is the hyperparameter that moves the
trainable-parameter count most. The original paper adapted only the attention query and value
projections; current practice adapts every linear layer in the block, attention and MLP alike, and
pays for it in trainable parameters. The only honest way to compare two presets is to build both
adapted models and ask each one what it is actually training — the count depends on the widths
inside the checkpoint, not on the length of the preset list. A rank-8 adapter on a narrow key
projection and on a wide MLP projection cost very different amounts.

**Problem Statement:**

Using the `transformers` and `peft` libraries, write a function
`lora_budget_comparison(model_id, r, preset_a, preset_b)`.

1. `preset_a` and `preset_b` are two lists of module names to adapt. Adapt the base model
   `model_id` twice, once under each preset, both at rank `r`. Build each `LoraConfig` the way the
   material does — decoder-only task type, `0.05` dropout, no bias training, and a scaling factor
   of twice the rank — then wrap the base model so the adapters are injected.
2. Ask each wrapped model for its own trainable and total parameter counts rather than deriving
   them from the config. Note that injecting adapters modifies the base model **in place**, so the
   second preset must be applied to a base model that has not already been adapted.
3. Report, for each preset, the number of trainable parameters and that number as a percentage of
   the wrapped model's total parameters, rounded to 4 decimal places.
4. Report the ratio of the two trainable counts, `preset_a` over `preset_b`, rounded to 4 decimal
   places.
5. Return a dictionary with exactly these five keys: `"trainable_a"`, `"pct_a"`, `"trainable_b"`,
   `"pct_b"` and `"ratio"`.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: model_id (string)
    Single line: a string
  Argument 2: r (integer)
    Single line: one integer
  Argument 3: preset_a (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 4: preset_b (list_string)
    Line 1: N (list length)
    Next N lines: one string each
```

**Output Format:**

For each test case, print the result of `lora_budget_comparison()` on a new line.

**Sample Input:**

```
model_id = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
r = 8
preset_a = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
preset_b = ["q_proj", "v_proj"]
```

**Sample Output:**

```
{'trainable_a': 2816, 'pct_a': 0.1158, 'trainable_b': 448, 'pct_b': 0.0184, 'ratio': 6.2857}
```

**Note:** The input handling is already in place — you only need to implement the function. The
models used here are tiny test checkpoints, so loading one inside the function is cheap. The two
percentages have different denominators, because a wrapped model's total includes the adapters it
just gained. The scaling factor plays no part in any number you report — it is specified so the
config matches the one the material builds, nothing more.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: lora_budget_comparison
RETURN TYPE: Dictionary

ARGUMENTS:
1. model_id → String
2. r        → Integer
3. preset_a → List of Strings
4. preset_b → List of Strings
```

---

### 3. Reference Solution Code

```python
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model


def lora_budget_comparison(model_id: str, r: int, preset_a: list, preset_b: list) -> dict:
    def budget(target_modules):
        model = AutoModelForCausalLM.from_pretrained(model_id)
        config = LoraConfig(task_type="CAUSAL_LM", r=r, lora_alpha=2 * r,
                            lora_dropout=0.05, bias="none",
                            target_modules=target_modules)
        wrapped = get_peft_model(model, config)
        return wrapped.get_nb_trainable_parameters()

    trainable_a, total_a = budget(preset_a)
    trainable_b, total_b = budget(preset_b)

    return {
        "trainable_a": int(trainable_a),
        "pct_a": round(100.0 * trainable_a / total_a, 4),
        "trainable_b": int(trainable_b),
        "pct_b": round(100.0 * trainable_b / total_b, 4),
        "ratio": round(trainable_a / trainable_b, 4),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
trl-internal-testing/tiny-Qwen2ForCausalLM-2.5
8
7
q_proj
k_proj
v_proj
o_proj
gate_proj
up_proj
down_proj
2
q_proj
v_proj
```

**out1.txt**
```
{'trainable_a': 2816, 'pct_a': 0.1158, 'trainable_b': 448, 'pct_b': 0.0184, 'ratio': 6.2857}
```

**in2.txt**
```
1
trl-internal-testing/small-Qwen2ForCausalLM-2.5
4
4
q_proj
k_proj
v_proj
o_proj
2
q_proj
v_proj
```

**out2.txt**
```
{'trainable_a': 7168, 'pct_a': 0.0184, 'trainable_b': 3584, 'pct_b': 0.0092, 'ratio': 2.0}
```

In [1]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model


def lora_budget_comparison(model_id: str, r: int, preset_a: list, preset_b: list) -> dict:
    def budget(target_modules):
        model = AutoModelForCausalLM.from_pretrained(model_id)
        config = LoraConfig(task_type="CAUSAL_LM", r=r, lora_alpha=2 * r,
                            lora_dropout=0.05, bias="none",
                            target_modules=target_modules)
        wrapped = get_peft_model(model, config)
        return wrapped.get_nb_trainable_parameters()

    trainable_a, total_a = budget(preset_a)
    trainable_b, total_b = budget(preset_b)

    return {
        "trainable_a": int(trainable_a),
        "pct_a": round(100.0 * trainable_a / total_a, 4),
        "trainable_b": int(trainable_b),
        "pct_b": round(100.0 * trainable_b / total_b, 4),
        "ratio": round(trainable_a / trainable_b, 4),
    }

/Users/pulkitmangal/AI-eng-lecture-cq/.venv-phi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = 'trl-internal-testing/tiny-Qwen2ForCausalLM-2.5'
r = 8
preset_a = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
preset_b = ['q_proj', 'v_proj']

lora_budget_comparison(model_id, r, preset_a, preset_b)


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 21616.00it/s]


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 10675.55it/s]

{'trainable_a': 2816,
 'pct_a': 0.1158,
 'trainable_b': 448,
 'pct_b': 0.0184,
 'ratio': 6.2857}

In [3]:
model_id = 'trl-internal-testing/small-Qwen2ForCausalLM-2.5'
r = 4
preset_a = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
preset_b = ['q_proj', 'v_proj']

lora_budget_comparison(model_id, r, preset_a, preset_b)


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 9649.47it/s]


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 12816.46it/s]

{'trainable_a': 7168,
 'pct_a': 0.0184,
 'trainable_b': 3584,
 'pct_b': 0.0092,
 'ratio': 2.0}

### [Class 4] Question 2. Merging a LoRA Adapter Into the Base Weights with PEFT

**Context:** A LoRA-adapted layer keeps its pretrained matrix frozen and evaluates a second, low-rank
branch beside it on every forward pass. That branch is pure overhead once training is over, which is
why an adapter is *merged* before deployment: the scaled low-rank product is folded into the frozen
matrix and the two branches collapse into one. What comes out is an ordinary layer again — same
shape, same inference cost, no adapter machinery left anywhere in the model. The fold is a single
library call, and confirming it worked means looking at three things: a weight that moved, an
adapter that is gone, and a parameter count back at the base model's.

**Problem Statement:**

Using the `transformers` and `peft` libraries, write a function
`merge_lora_adapter(model_id, r, lora_alpha, target_modules, a_fill, b_fill, probe_row, probe_col)`.

1. Load the base causal language model `model_id` in 32-bit floating point, build a `LoraConfig`
   for a decoder-only model at rank `r` with scaling factor `lora_alpha`, `0.05` dropout, no bias
   training and `target_modules` as the modules to adapt, then wrap the model so the adapters are
   injected.
2. The probe throughout is the **query projection of the first transformer block**. Before anything
   is merged, read the frozen weight sitting at row `probe_row`, column `probe_col` of that
   projection.
3. A freshly injected adapter contributes nothing — one of its two matrices is initialised to zeros,
   so merging it would change no weight at all. Stand in for a trained adapter by filling that
   projection's down-projection matrix entirely with `a_fill` and its up-projection matrix entirely
   with `b_fill`.
4. Merge the adapters into the base weights and unload the adapter machinery, using the library call
   that does both. Then read the same (`probe_row`, `probe_col`) entry off the merged model.
5. Report how many modules in the merged model still carry an injected down-projection matrix, and
   the merged model's total parameter count.
6. Return a dictionary with exactly these five keys: `"base_entry"`, `"merged_entry"` and `"delta"`
   — the entry before, the entry after, and the difference, each rounded to 6 decimal places —
   plus `"adapters_left"` and `"merged_params"` as integers.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: model_id (string)
    Single line: a string
  Argument 2: r (integer)
    Single line: one integer
  Argument 3: lora_alpha (integer)
    Single line: one integer
  Argument 4: target_modules (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 5: a_fill (float)
    Single line: one float
  Argument 6: b_fill (float)
    Single line: one float
  Argument 7: probe_row (integer)
    Single line: one integer
  Argument 8: probe_col (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `merge_lora_adapter()` on a new line.

**Sample Input:**

```
model_id = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
r = 4
lora_alpha = 8
target_modules = ["q_proj", "v_proj"]
a_fill = 0.01
b_fill = 0.02
probe_row = 2
probe_col = 3
```

**Sample Output:**

```
{'base_entry': 0.128906, 'merged_entry': 0.130506, 'delta': 0.0016, 'adapters_left': 0, 'merged_params': 2428632}
```

**Note:** The input handling is already in place — you only need to implement the function. The
models are tiny test checkpoints, so loading one inside the function is cheap. Read both entries off
the model rather than reconstructing either: the frozen weight is a property of the checkpoint, and
the merged one is whatever the library wrote back. Note that merging returns a model whose module
layout is the plain base architecture again, not the wrapped one — the path to the query projection
is not the same before and after.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: merge_lora_adapter
RETURN TYPE: Dictionary

ARGUMENTS:
1. model_id       → String
2. r              → Integer
3. lora_alpha     → Integer
4. target_modules → List of Strings
5. a_fill         → Float
6. b_fill         → Float
7. probe_row      → Integer
8. probe_col      → Integer
```

---

### 3. Reference Solution Code

```python
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model


def merge_lora_adapter(model_id: str, r: int, lora_alpha: int, target_modules: list,
                       a_fill: float, b_fill: float, probe_row: int,
                       probe_col: int) -> dict:
    base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
    config = LoraConfig(task_type="CAUSAL_LM", r=r, lora_alpha=lora_alpha,
                        lora_dropout=0.05, bias="none", target_modules=target_modules)
    peft_model = get_peft_model(base, config)

    layer = peft_model.base_model.model.model.layers[0].self_attn.q_proj
    base_entry = float(layer.base_layer.weight[probe_row, probe_col])

    with torch.no_grad():
        layer.lora_A["default"].weight.fill_(a_fill)
        layer.lora_B["default"].weight.fill_(b_fill)

    merged_model = peft_model.merge_and_unload()
    merged_layer = merged_model.model.layers[0].self_attn.q_proj
    merged_entry = float(merged_layer.weight[probe_row, probe_col])

    adapters_left = sum(1 for _, m in merged_model.named_modules() if hasattr(m, "lora_A"))

    return {
        "base_entry": round(base_entry, 6),
        "merged_entry": round(merged_entry, 6),
        "delta": round(merged_entry - base_entry, 6),
        "adapters_left": adapters_left,
        "merged_params": sum(p.numel() for p in merged_model.parameters()),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
trl-internal-testing/tiny-Qwen2ForCausalLM-2.5
4
8
2
q_proj
v_proj
0.01
0.02
2
3
```

**out1.txt**
```
{'base_entry': 0.128906, 'merged_entry': 0.130506, 'delta': 0.0016, 'adapters_left': 0, 'merged_params': 2428632}
```

**in2.txt**
```
1
trl-internal-testing/small-Qwen2ForCausalLM-2.5
8
24
1
q_proj
0.05
-0.02
5
11
```

**out2.txt**
```
{'base_entry': -0.004089, 'merged_entry': -0.028089, 'delta': -0.024, 'adapters_left': 0, 'merged_params': 38950272}
```

In [4]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model


def merge_lora_adapter(model_id: str, r: int, lora_alpha: int, target_modules: list,
                       a_fill: float, b_fill: float, probe_row: int,
                       probe_col: int) -> dict:
    base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
    config = LoraConfig(task_type="CAUSAL_LM", r=r, lora_alpha=lora_alpha,
                        lora_dropout=0.05, bias="none", target_modules=target_modules)
    peft_model = get_peft_model(base, config)

    layer = peft_model.base_model.model.model.layers[0].self_attn.q_proj
    base_entry = float(layer.base_layer.weight[probe_row, probe_col])

    with torch.no_grad():
        layer.lora_A["default"].weight.fill_(a_fill)
        layer.lora_B["default"].weight.fill_(b_fill)

    merged_model = peft_model.merge_and_unload()
    merged_layer = merged_model.model.layers[0].self_attn.q_proj
    merged_entry = float(merged_layer.weight[probe_row, probe_col])

    adapters_left = sum(1 for _, m in merged_model.named_modules() if hasattr(m, "lora_A"))

    return {
        "base_entry": round(base_entry, 6),
        "merged_entry": round(merged_entry, 6),
        "delta": round(merged_entry - base_entry, 6),
        "adapters_left": adapters_left,
        "merged_params": sum(p.numel() for p in merged_model.parameters()),
    }

In [5]:
model_id = 'trl-internal-testing/tiny-Qwen2ForCausalLM-2.5'
r = 4
lora_alpha = 8
target_modules = ['q_proj', 'v_proj']
a_fill = 0.01
b_fill = 0.02
probe_row = 2
probe_col = 3

merge_lora_adapter(model_id, r, lora_alpha, target_modules, a_fill, b_fill, probe_row, probe_col)


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 4755.05it/s]

{'base_entry': 0.128906,
 'merged_entry': 0.130506,
 'delta': 0.0016,
 'adapters_left': 0,
 'merged_params': 2428632}

In [6]:
model_id = 'trl-internal-testing/small-Qwen2ForCausalLM-2.5'
r = 8
lora_alpha = 24
target_modules = ['q_proj']
a_fill = 0.05
b_fill = -0.02
probe_row = 5
probe_col = 11

merge_lora_adapter(model_id, r, lora_alpha, target_modules, a_fill, b_fill, probe_row, probe_col)


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 340.51it/s]

{'base_entry': -0.004089,
 'merged_entry': -0.028089,
 'delta': -0.024,
 'adapters_left': 0,
 'merged_params': 38950272}

### [Class 4] Question 3. Supervised Fine-Tuning Loss on the Response Tokens Only

**Context:** A supervised fine-tuning example is a single sequence holding a prompt and the response
the model should have produced, and it is trained with the very objective used in pretraining: at
each position the model predicts the following token, and the penalty is the negative log of the
probability it gave the token that actually came next. The one thing that changes is what counts.
The prompt is context, not a target — it flows through the forward pass so the model can condition
on it, but it contributes nothing to the loss. Only the response is scored, which is what teaches a
model to *answer* an instruction rather than to *reproduce* it. Getting that accounting right is the
difference between fine-tuning an assistant and fine-tuning a parrot.

**Problem Statement:**

Using `torch`, write a function `sft_response_loss(logits, prompt_ids, response_ids)`.

1. The training sequence is `prompt_ids` followed by `response_ids`, one token id per element.
2. `logits` holds the model's raw scores for that sequence: one row per position in the sequence,
   one column per vocabulary entry. Row `t` scores the candidates for the token that comes *after*
   position `t`, so the last row's prediction has no target inside the sequence.
3. Score a single prediction with the usual next-token cross-entropy: the negative natural
   logarithm of the softmax probability that row assigned to the token which actually followed.
4. Only the response is scored. The loss has exactly one term per token in `response_ids` — each of
   those tokens, scored at the position that predicts it. Prompt tokens contribute no term of their
   own.
5. Return the mean of those terms, rounded to 4 decimal places.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: logits (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 2: prompt_ids (list_int)
    Line 1: P (list length)
    Line 2: P space-separated integers
  Argument 3: response_ids (list_int)
    Line 1: N (list length)
    Line 2: N space-separated integers
```

**Output Format:**

For each test case, print the result of `sft_response_loss()` on a new line.

**Sample Input:**

```
logits = tensor([[-1.1,  0.0,  1.5, -1.0,  0.3,  0.6,  4.7, -1.1],
                 [ 0.2,  6.3, -0.7,  0.8, -0.7, -0.1,  0.8,  0.0],
                 [-1.2,  0.8,  0.9,  0.8,  1.1,  0.7,  1.0,  4.4],
                 [ 1.0, -1.3,  4.6, -0.2, -0.7,  0.4,  1.1, -1.0],
                 [-0.8, -0.5,  1.2, -1.3,  1.5,  0.9, -0.2, -0.2],
                 [ 0.0,  1.0, -0.1, -0.9, -0.8,  0.2,  0.3, -0.8],
                 [ 0.3, -0.9,  0.0,  0.7,  2.3, -0.4, -0.1,  0.5],
                 [-0.7,  0.1,  1.0,  0.0,  1.0, -1.4, -1.3,  0.4],
                 [ 0.8,  0.9, -1.1,  0.6, -0.3,  0.4,  1.4, -1.5]])
prompt_ids   = [3, 6, 1, 7, 2]
response_ids = [5, 0, 4, 7]
```

**Sample Output:**

```
1.5796
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is loaded and nothing is downloaded: the score matrix stands in for a forward pass that has already
run, and it arrives as a `float32` tensor, so keep the computation in `torch`. How you keep the
prompt positions out of the average is entirely your choice — any approach is fine as long as the
terms that end up counted are the ones step 4 describes. Reading the sample is a useful sanity
check on the accounting: the prompt's own tokens are predicted very confidently here, so a figure
noticeably below the sample's is a sign that positions which should not count are counting.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: sft_response_loss
RETURN TYPE: Float

ARGUMENTS:
1. logits       → Torch 2D Tensor
2. prompt_ids   → List of Ints
3. response_ids → List of Ints
```

---

### 3. Reference Solution Code

```python
import torch


def sft_response_loss(logits: torch.Tensor, prompt_ids: list, response_ids: list) -> float:
    token_ids = prompt_ids + response_ids
    log_probs = torch.log_softmax(logits, dim=-1)
    first = len(prompt_ids) - 1
    terms = [-log_probs[t, token_ids[t + 1]] for t in range(first, len(token_ids) - 1)]
    return round(float(sum(terms) / len(terms)), 4)
```

---

### 4. Test Cases

**in1.txt**
```
1
9 8
-1.1 0.0 1.5 -1.0 0.3 0.6 4.7 -1.1
0.2 6.3 -0.7 0.8 -0.7 -0.1 0.8 0.0
-1.2 0.8 0.9 0.8 1.1 0.7 1.0 4.4
1.0 -1.3 4.6 -0.2 -0.7 0.4 1.1 -1.0
-0.8 -0.5 1.2 -1.3 1.5 0.9 -0.2 -0.2
0.0 1.0 -0.1 -0.9 -0.8 0.2 0.3 -0.8
0.3 -0.9 0.0 0.7 2.3 -0.4 -0.1 0.5
-0.7 0.1 1.0 0.0 1.0 -1.4 -1.3 0.4
0.8 0.9 -1.1 0.6 -0.3 0.4 1.4 -1.5
5
3 6 1 7 2
4
5 0 4 7
```

**out1.txt**
```
1.5796
```

**in2.txt**
```
1
12 10
-0.2 -0.6 5.0 -0.4 1.4 -1.2 -0.6 1.3 0.3 -1.1
0.5 -1.2 0.9 0.2 -0.1 -0.8 -0.8 -0.5 -0.1 3.9
0.4 -1.0 -0.6 -0.9 0.1 4.1 1.2 1.0 1.3 -0.6
-0.1 2.1 -0.5 0.5 0.5 -0.9 -0.1 -1.2 0.6 0.8
0.2 -1.3 -0.6 0.9 -0.7 0.2 -0.2 -0.2 -0.9 -1.3
-0.8 -0.6 1.3 2.9 0.6 0.8 0.5 -0.7 0.7 -0.1
-1.0 -0.2 0.8 -0.2 0.1 0.4 3.5 -1.1 -0.8 0.8
1.4 -0.5 0.9 0.4 1.1 0.6 -0.1 1.2 -0.7 -0.2
0.5 0.8 -1.4 -0.1 1.2 1.4 0.8 3.4 0.4 -1.0
0.3 -0.6 0.1 -1.2 1.5 -0.2 0.5 -1.1 0.5 0.5
0.9 1.1 -0.1 -0.5 -1.2 -0.2 1.2 1.4 -0.5 2.3
-0.8 -0.2 -0.8 -0.5 -1.4 0.2 0.8 0.3 -0.9 0.7
4
8 2 9 5
8
1 6 3 6 0 7 4 9
```

**out2.txt**
```
1.0083
```

In [7]:
import torch


def sft_response_loss(logits: torch.Tensor, prompt_ids: list, response_ids: list) -> float:
    token_ids = prompt_ids + response_ids
    log_probs = torch.log_softmax(logits, dim=-1)
    first = len(prompt_ids) - 1
    terms = [-log_probs[t, token_ids[t + 1]] for t in range(first, len(token_ids) - 1)]
    return round(float(sum(terms) / len(terms)), 4)

In [8]:
logits = torch.tensor([
    [-1.1, 0.0, 1.5, -1.0, 0.3, 0.6, 4.7, -1.1],
    [0.2, 6.3, -0.7, 0.8, -0.7, -0.1, 0.8, 0.0],
    [-1.2, 0.8, 0.9, 0.8, 1.1, 0.7, 1.0, 4.4],
    [1.0, -1.3, 4.6, -0.2, -0.7, 0.4, 1.1, -1.0],
    [-0.8, -0.5, 1.2, -1.3, 1.5, 0.9, -0.2, -0.2],
    [0.0, 1.0, -0.1, -0.9, -0.8, 0.2, 0.3, -0.8],
    [0.3, -0.9, 0.0, 0.7, 2.3, -0.4, -0.1, 0.5],
    [-0.7, 0.1, 1.0, 0.0, 1.0, -1.4, -1.3, 0.4],
    [0.8, 0.9, -1.1, 0.6, -0.3, 0.4, 1.4, -1.5]
], dtype=torch.float32)
prompt_ids = [3, 6, 1, 7, 2]
response_ids = [5, 0, 4, 7]

sft_response_loss(logits, prompt_ids, response_ids)

1.5796

In [9]:
logits = torch.tensor([
    [-0.2, -0.6, 5.0, -0.4, 1.4, -1.2, -0.6, 1.3, 0.3, -1.1],
    [0.5, -1.2, 0.9, 0.2, -0.1, -0.8, -0.8, -0.5, -0.1, 3.9],
    [0.4, -1.0, -0.6, -0.9, 0.1, 4.1, 1.2, 1.0, 1.3, -0.6],
    [-0.1, 2.1, -0.5, 0.5, 0.5, -0.9, -0.1, -1.2, 0.6, 0.8],
    [0.2, -1.3, -0.6, 0.9, -0.7, 0.2, -0.2, -0.2, -0.9, -1.3],
    [-0.8, -0.6, 1.3, 2.9, 0.6, 0.8, 0.5, -0.7, 0.7, -0.1],
    [-1.0, -0.2, 0.8, -0.2, 0.1, 0.4, 3.5, -1.1, -0.8, 0.8],
    [1.4, -0.5, 0.9, 0.4, 1.1, 0.6, -0.1, 1.2, -0.7, -0.2],
    [0.5, 0.8, -1.4, -0.1, 1.2, 1.4, 0.8, 3.4, 0.4, -1.0],
    [0.3, -0.6, 0.1, -1.2, 1.5, -0.2, 0.5, -1.1, 0.5, 0.5],
    [0.9, 1.1, -0.1, -0.5, -1.2, -0.2, 1.2, 1.4, -0.5, 2.3],
    [-0.8, -0.2, -0.8, -0.5, -1.4, 0.2, 0.8, 0.3, -0.9, 0.7]
], dtype=torch.float32)
prompt_ids = [8, 2, 9, 5]
response_ids = [1, 6, 3, 6, 0, 7, 4, 9]

sft_response_loss(logits, prompt_ids, response_ids)

1.0083

### [Class 5] Question 4. Telling Voiced From Unvoiced With Two Cheap Features

**Context:** Before any deep learning touches a waveform, two frame-level features already say a
great deal about it. Chop the signal into short overlapping windows and, for each one, measure how
loud it is and how often it crosses zero. A vowel is a periodic wave — loud, and crossing zero only
as often as its pitch requires. A fricative like "sss" is turbulent airflow — comparable loudness,
but crossing zero constantly. That contrast is enough to separate speech types with a threshold and
no training at all, which is why voice-activity detectors have used it for decades.

**Problem Statement:**

Using `numpy`, write a function
`voiced_unvoiced_report(signal, sample_rate, frame_ms, hop_ms, zcr_threshold)`.

1. Convert the frame length and the hop from milliseconds to whole numbers of samples at
   `sample_rate`, truncating any fraction. Report both.
2. Cut `signal` into frames of that length, advancing by that hop. Frames start at sample `0` and a
   new frame is taken at every hop position **strictly before** `len(signal) − frame_length`; no
   partial frame at the tail is kept. Report how many frames that yields.
3. For each frame compute its **RMS energy** — the square root of the mean of its squared samples —
   and its **zero-crossing rate** — the fraction of adjacent sample pairs within the frame whose
   sign differs. Report the mean RMS across frames and the largest zero-crossing rate any frame
   reached, each rounded to 6 decimal places.
4. A frame counts as **unvoiced** when its zero-crossing rate is strictly greater than
   `zcr_threshold`. Report how many frames are unvoiced, and the index of the first one — or `-1`
   when no frame is.
5. Return a dictionary with exactly these seven keys: `"num_frames"`, `"frame_len"`, `"hop_len"`,
   `"unvoiced_frames"` and `"first_unvoiced_frame"` as integers, and `"mean_rms"` and `"max_zcr"`
   as floats.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: signal (numpy_1d)
    Line 1: N (array length)
    Line 2: N space-separated values
  Argument 2: sample_rate (integer)
    Single line: one integer
  Argument 3: frame_ms (float)
    Single line: one float
  Argument 4: hop_ms (float)
    Single line: one float
  Argument 5: zcr_threshold (float)
    Single line: one float
```

**Output Format:**

For each test case, print the result of `voiced_unvoiced_report()` on a new line.

**Sample Input:**

```
signal = np.array([0.0, 0.2724, 0.4854, ...])   # 120 samples: a low tone, then noise
sample_rate   = 800
frame_ms      = 25.0
hop_ms        = 10.0
zcr_threshold = 0.15
```

**Sample Output:**

```
{'num_frames': 13, 'frame_len': 20, 'hop_len': 8, 'mean_rms': 0.304009, 'max_zcr': 0.526316, 'unvoiced_frames': 7, 'first_unvoiced_frame': 6}
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is loaded and no audio file is read. Sign changes are counted between neighbouring samples *inside*
one frame, so a frame of `L` samples offers `L − 1` pairs to compare — the rate is over those pairs,
not over `L`. The sample rates in the test cases are lower than the 16 kHz speech standard purely to
keep the payload readable; the millisecond-to-sample conversion is unchanged by that.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: voiced_unvoiced_report
RETURN TYPE: Dictionary

ARGUMENTS:
1. signal        → Numpy 1D Array
2. sample_rate   → Integer
3. frame_ms      → Float
4. hop_ms        → Float
5. zcr_threshold → Float
```

---

### 3. Reference Solution Code

```python
import numpy as np


def voiced_unvoiced_report(signal: np.ndarray, sample_rate: int, frame_ms: float,
                           hop_ms: float, zcr_threshold: float) -> dict:
    frame_len = int(frame_ms / 1000 * sample_rate)
    hop = int(hop_ms / 1000 * sample_rate)

    frames = [signal[i:i + frame_len]
              for i in range(0, len(signal) - frame_len, hop)]

    rms = np.array([np.sqrt(np.mean(f ** 2)) for f in frames])
    zcr = np.array([np.mean(np.diff(np.sign(f)) != 0) for f in frames])

    unvoiced = zcr > zcr_threshold

    return {
        "num_frames": len(frames),
        "frame_len": frame_len,
        "hop_len": hop,
        "mean_rms": round(float(rms.mean()), 6),
        "max_zcr": round(float(zcr.max()), 6),
        "unvoiced_frames": int(unvoiced.sum()),
        "first_unvoiced_frame": int(np.argmax(unvoiced)) if unvoiced.any() else -1,
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
120
0.0 0.0471 0.0939 0.1401 0.1854 0.2296 0.2724 0.3135 0.3527 0.3897 0.4243 0.4562 0.4854 0.5116 0.5346 0.5543 0.5706 0.5834 0.5926 0.5982 0.6 0.5982 0.5926 0.5834 0.5706 0.5543 0.5346 0.5116 0.4854 0.4562 0.4243 0.3897 0.3527 0.3135 0.2724 0.2296 0.1854 0.1401 0.0939 0.0471 0.0 -0.0471 -0.0939 -0.1401 -0.1854 -0.2296 -0.2724 -0.3135 -0.3527 -0.3897 -0.4243 -0.4562 -0.4854 -0.5116 -0.5346 -0.5543 -0.5706 -0.5834 -0.5926 -0.5982 0.0609 -0.208 0.1501 0.1881 -0.3902 -0.2604 0.0256 -0.0632 -0.0034 -0.1706 0.1759 0.1556 0.0132 0.2254 0.0935 -0.1719 0.0738 -0.1918 0.1757 -0.01 -0.037 -0.1362 0.2445 -0.0309 -0.0857 -0.0704 0.1065 0.0731 0.0825 0.0862 0.4283 -0.0813 -0.1024 -0.1628 0.1232 0.2258 -0.0228 -0.168 -0.1649 0.1301 0.1487 0.1086 -0.1331 0.0464 0.0233 0.0437 0.1743 0.0447 0.1358 0.0135 0.0578 0.1263 -0.2914 -0.0639 -0.0941 -0.1278 -0.055 0.299 -0.1732 0.1937
800
25.0
10.0
0.15
```

**out1.txt**
```
{'num_frames': 13, 'frame_len': 20, 'hop_len': 8, 'mean_rms': 0.304009, 'max_zcr': 0.526316, 'unvoiced_frames': 7, 'first_unvoiced_frame': 6}
```

**in2.txt**
```
1
200
0.0 0.0353 0.0704 0.1051 0.1391 0.1722 0.2043 0.2351 0.2645 0.2923 0.3182 0.3422 0.3641 0.3837 0.401 0.4157 0.428 0.4376 0.4445 0.4486 0.45 0.4486 0.4445 0.4376 0.428 0.4157 0.401 0.3837 0.3641 0.3422 0.3182 0.2923 0.2645 0.2351 0.2043 0.1722 0.1391 0.1051 0.0704 0.0353 0.0 -0.0353 -0.0704 -0.1051 -0.1391 -0.1722 -0.2043 -0.2351 -0.2645 -0.2923 -0.3182 -0.3422 -0.3641 -0.3837 -0.401 -0.4157 -0.428 -0.4376 -0.4445 -0.4486 -0.45 -0.4486 -0.4445 -0.4376 -0.428 -0.4157 -0.401 -0.3837 -0.3641 -0.3422 -0.3182 -0.2923 -0.2645 -0.2351 -0.2043 -0.1722 -0.1391 -0.1051 -0.0704 -0.0353 -0.0 0.0353 0.0704 0.1051 0.1391 0.1722 0.2043 0.2351 0.2645 0.2923 0.3182 0.3422 0.3641 0.3837 0.401 0.4157 0.428 0.4376 0.4445 0.4486 0.45 0.4486 0.4445 0.4376 0.428 0.4157 0.401 0.3837 0.3641 0.3422 0.3182 0.2923 0.2645 0.2351 0.2043 0.1722 0.1391 0.1051 0.0704 0.0353 0.0 -0.0353 -0.0704 -0.1051 -0.1391 -0.1722 -0.2043 -0.2351 -0.2645 -0.2923 -0.3182 -0.3422 -0.3641 -0.3837 -0.401 -0.4157 -0.428 -0.4376 -0.4445 -0.4486 -0.45 -0.4486 -0.4445 -0.4376 -0.428 -0.4157 -0.401 -0.3837 -0.3641 -0.3422 -0.3182 -0.2923 -0.2645 -0.2351 -0.2043 -0.1722 -0.1391 -0.1051 -0.0704 -0.0353 -0.0 0.0353 0.0704 0.1051 0.1391 0.1722 0.2043 0.2351 0.2645 0.2923 0.3182 0.3422 0.3641 0.3837 0.401 0.4157 0.428 0.4376 0.4445 0.4486 0.45 0.4486 0.4445 0.4376 0.428 0.4157 0.401 0.3837 0.3641 0.3422 0.3182 0.2923 0.2645 0.2351 0.2043 0.1722 0.1391 0.1051 0.0704 0.0353
1600
25.0
10.0
0.10
```

**out2.txt**
```
{'num_frames': 10, 'frame_len': 40, 'hop_len': 16, 'mean_rms': 0.318211, 'max_zcr': 0.051282, 'unvoiced_frames': 0, 'first_unvoiced_frame': -1}
```

In [10]:
import numpy as np


def voiced_unvoiced_report(signal: np.ndarray, sample_rate: int, frame_ms: float,
                           hop_ms: float, zcr_threshold: float) -> dict:
    frame_len = int(frame_ms / 1000 * sample_rate)
    hop = int(hop_ms / 1000 * sample_rate)

    frames = [signal[i:i + frame_len]
              for i in range(0, len(signal) - frame_len, hop)]

    rms = np.array([np.sqrt(np.mean(f ** 2)) for f in frames])
    zcr = np.array([np.mean(np.diff(np.sign(f)) != 0) for f in frames])

    unvoiced = zcr > zcr_threshold

    return {
        "num_frames": len(frames),
        "frame_len": frame_len,
        "hop_len": hop,
        "mean_rms": round(float(rms.mean()), 6),
        "max_zcr": round(float(zcr.max()), 6),
        "unvoiced_frames": int(unvoiced.sum()),
        "first_unvoiced_frame": int(np.argmax(unvoiced)) if unvoiced.any() else -1,
    }

In [11]:
signal = np.array([
    0.0, 0.0471, 0.0939, 0.1401, 0.1854, 0.2296, 0.2724, 0.3135, 0.3527, 0.3897, 0.4243,
    0.4562, 0.4854, 0.5116, 0.5346, 0.5543, 0.5706, 0.5834, 0.5926, 0.5982, 0.6, 0.5982,
    0.5926, 0.5834, 0.5706, 0.5543, 0.5346, 0.5116, 0.4854, 0.4562, 0.4243, 0.3897, 0.3527,
    0.3135, 0.2724, 0.2296, 0.1854, 0.1401, 0.0939, 0.0471, 0.0, -0.0471, -0.0939, -0.1401,
    -0.1854, -0.2296, -0.2724, -0.3135, -0.3527, -0.3897, -0.4243, -0.4562, -0.4854,
    -0.5116, -0.5346, -0.5543, -0.5706, -0.5834, -0.5926, -0.5982, 0.0609, -0.208, 0.1501,
    0.1881, -0.3902, -0.2604, 0.0256, -0.0632, -0.0034, -0.1706, 0.1759, 0.1556, 0.0132,
    0.2254, 0.0935, -0.1719, 0.0738, -0.1918, 0.1757, -0.01, -0.037, -0.1362, 0.2445,
    -0.0309, -0.0857, -0.0704, 0.1065, 0.0731, 0.0825, 0.0862, 0.4283, -0.0813, -0.1024,
    -0.1628, 0.1232, 0.2258, -0.0228, -0.168, -0.1649, 0.1301, 0.1487, 0.1086, -0.1331,
    0.0464, 0.0233, 0.0437, 0.1743, 0.0447, 0.1358, 0.0135, 0.0578, 0.1263, -0.2914,
    -0.0639, -0.0941, -0.1278, -0.055, 0.299, -0.1732, 0.1937
])
sample_rate = 800
frame_ms = 25.0
hop_ms = 10.0
zcr_threshold = 0.15

voiced_unvoiced_report(signal, sample_rate, frame_ms, hop_ms, zcr_threshold)

{'num_frames': 13,
 'frame_len': 20,
 'hop_len': 8,
 'mean_rms': 0.304009,
 'max_zcr': 0.526316,
 'unvoiced_frames': 7,
 'first_unvoiced_frame': 6}

In [12]:
signal = np.array([
    0.0, 0.0353, 0.0704, 0.1051, 0.1391, 0.1722, 0.2043, 0.2351, 0.2645, 0.2923, 0.3182,
    0.3422, 0.3641, 0.3837, 0.401, 0.4157, 0.428, 0.4376, 0.4445, 0.4486, 0.45, 0.4486,
    0.4445, 0.4376, 0.428, 0.4157, 0.401, 0.3837, 0.3641, 0.3422, 0.3182, 0.2923, 0.2645,
    0.2351, 0.2043, 0.1722, 0.1391, 0.1051, 0.0704, 0.0353, 0.0, -0.0353, -0.0704, -0.1051,
    -0.1391, -0.1722, -0.2043, -0.2351, -0.2645, -0.2923, -0.3182, -0.3422, -0.3641,
    -0.3837, -0.401, -0.4157, -0.428, -0.4376, -0.4445, -0.4486, -0.45, -0.4486, -0.4445,
    -0.4376, -0.428, -0.4157, -0.401, -0.3837, -0.3641, -0.3422, -0.3182, -0.2923, -0.2645,
    -0.2351, -0.2043, -0.1722, -0.1391, -0.1051, -0.0704, -0.0353, -0.0, 0.0353, 0.0704,
    0.1051, 0.1391, 0.1722, 0.2043, 0.2351, 0.2645, 0.2923, 0.3182, 0.3422, 0.3641, 0.3837,
    0.401, 0.4157, 0.428, 0.4376, 0.4445, 0.4486, 0.45, 0.4486, 0.4445, 0.4376, 0.428,
    0.4157, 0.401, 0.3837, 0.3641, 0.3422, 0.3182, 0.2923, 0.2645, 0.2351, 0.2043, 0.1722,
    0.1391, 0.1051, 0.0704, 0.0353, 0.0, -0.0353, -0.0704, -0.1051, -0.1391, -0.1722,
    -0.2043, -0.2351, -0.2645, -0.2923, -0.3182, -0.3422, -0.3641, -0.3837, -0.401, -0.4157,
    -0.428, -0.4376, -0.4445, -0.4486, -0.45, -0.4486, -0.4445, -0.4376, -0.428, -0.4157,
    -0.401, -0.3837, -0.3641, -0.3422, -0.3182, -0.2923, -0.2645, -0.2351, -0.2043, -0.1722,
    -0.1391, -0.1051, -0.0704, -0.0353, -0.0, 0.0353, 0.0704, 0.1051, 0.1391, 0.1722,
    0.2043, 0.2351, 0.2645, 0.2923, 0.3182, 0.3422, 0.3641, 0.3837, 0.401, 0.4157, 0.428,
    0.4376, 0.4445, 0.4486, 0.45, 0.4486, 0.4445, 0.4376, 0.428, 0.4157, 0.401, 0.3837,
    0.3641, 0.3422, 0.3182, 0.2923, 0.2645, 0.2351, 0.2043, 0.1722, 0.1391, 0.1051, 0.0704,
    0.0353
])
sample_rate = 1600
frame_ms = 25.0
hop_ms = 10.0
zcr_threshold = 0.10

voiced_unvoiced_report(signal, sample_rate, frame_ms, hop_ms, zcr_threshold)

{'num_frames': 10,
 'frame_len': 40,
 'hop_len': 16,
 'mean_rms': 0.318211,
 'max_zcr': 0.051282,
 'unvoiced_frames': 0,
 'first_unvoiced_frame': -1}

### [Class 5] Question 5. The Mel Spectrogram the Model Actually Sees

**Context:** A neural network is not handed 48,000 raw samples and asked to find structure in them.
The waveform is first turned into a two-dimensional image: overlapping short windows along time,
each decomposed into frequencies, those frequencies squashed onto a perceptual scale and
log-compressed. What comes out is a compact matrix — frequency bands down one axis, time frames
along the other — and that matrix, not the waveform, is what a speech encoder consumes. Knowing how
its shape follows from the window and hop settings is how you predict a model's input size and its
memory cost before you run anything.

**Problem Statement:**

Using the `numpy` and `librosa` libraries, write a function
`mel_spectrogram_report(sample_rate, duration_s, tone_hz, n_mels, n_fft, hop_length)`.

1. Synthesise the input waveform: take `sample_rate × duration_s` samples, truncated to a whole
   number, at times `k / sample_rate` for `k = 0, 1, …`; the sample at each time is
   `0.5 · sin(2π · tone_hz · t)`. Report how many samples that is.
2. Compute the Mel spectrogram of that waveform with `librosa`, at `n_mels` bands, an FFT window of
   `n_fft` and a hop of `hop_length`, then convert it to decibels relative to its own maximum.
3. Report the two dimensions of the result — the number of bands and the number of time frames —
   and their product, the number of cells the model receives.
4. Report the ratio of raw samples to cells, rounded to 4 decimal places: how much smaller the
   representation is than the waveform it came from. Report also how many frames the settings
   produce per second of audio, rounded to 4 decimal places.
5. Report which band is loudest overall — the 0-based index of the band whose mean decibel value
   across all time frames is highest.
6. Return a dictionary with exactly these seven keys: `"n_mels"`, `"n_frames"`, `"raw_samples"`,
   `"mel_cells"` and `"loudest_band"` as integers, and `"compression_ratio"` and
   `"frames_per_second"` as floats.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: sample_rate (integer)
    Single line: one integer
  Argument 2: duration_s (float)
    Single line: one float
  Argument 3: tone_hz (float)
    Single line: one float
  Argument 4: n_mels (integer)
    Single line: one integer
  Argument 5: n_fft (integer)
    Single line: one integer
  Argument 6: hop_length (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `mel_spectrogram_report()` on a new line.

**Sample Input:**

```
sample_rate = 16000
duration_s  = 3.0
tone_hz     = 440.0
n_mels      = 80
n_fft       = 400
hop_length  = 160
```

**Sample Output:**

```
{'n_mels': 80, 'n_frames': 301, 'raw_samples': 48000, 'mel_cells': 24080, 'compression_ratio': 1.9934, 'frames_per_second': 100.0, 'loudest_band': 11}
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is downloaded; the waveform is built from the arguments. Leave the mel and decibel conversions at
their library defaults apart from the three settings named above — the frame count in particular is
the library's own, so derive it from the result rather than predicting it. Pass the sample rate to
the mel call: it decides where the perceptual bands sit, and therefore which band comes out loudest.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: mel_spectrogram_report
RETURN TYPE: Dictionary

ARGUMENTS:
1. sample_rate → Integer
2. duration_s  → Float
3. tone_hz     → Float
4. n_mels      → Integer
5. n_fft       → Integer
6. hop_length  → Integer
```

---

### 3. Reference Solution Code

```python
import librosa
import numpy as np


def mel_spectrogram_report(sample_rate: int, duration_s: float, tone_hz: float,
                           n_mels: int, n_fft: int, hop_length: int) -> dict:
    n = int(duration_s * sample_rate)
    t = np.arange(n) / sample_rate
    y = (0.5 * np.sin(2 * np.pi * tone_hz * t)).astype(np.float32)

    S = librosa.feature.melspectrogram(y=y, sr=sample_rate, n_mels=n_mels,
                                       n_fft=n_fft, hop_length=hop_length)
    S_db = librosa.power_to_db(S, ref=np.max)

    n_bands, n_frames = S.shape

    return {
        "n_mels": int(n_bands),
        "n_frames": int(n_frames),
        "raw_samples": n,
        "mel_cells": int(n_bands * n_frames),
        "compression_ratio": round(n / (n_bands * n_frames), 4),
        "frames_per_second": round(sample_rate / hop_length, 4),
        "loudest_band": int(np.argmax(S_db.mean(axis=1))),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
16000
3.0
440.0
80
400
160
```

**out1.txt**
```
{'n_mels': 80, 'n_frames': 301, 'raw_samples': 48000, 'mel_cells': 24080, 'compression_ratio': 1.9934, 'frames_per_second': 100.0, 'loudest_band': 11}
```

**in2.txt**
```
1
8000
2.0
300.0
32
256
128
```

**out2.txt**
```
{'n_mels': 32, 'n_frames': 126, 'raw_samples': 16000, 'mel_cells': 4032, 'compression_ratio': 3.9683, 'frames_per_second': 62.5, 'loudest_band': 3}
```

In [13]:
import librosa
import numpy as np


def mel_spectrogram_report(sample_rate: int, duration_s: float, tone_hz: float,
                           n_mels: int, n_fft: int, hop_length: int) -> dict:
    n = int(duration_s * sample_rate)
    t = np.arange(n) / sample_rate
    y = (0.5 * np.sin(2 * np.pi * tone_hz * t)).astype(np.float32)

    S = librosa.feature.melspectrogram(y=y, sr=sample_rate, n_mels=n_mels,
                                       n_fft=n_fft, hop_length=hop_length)
    S_db = librosa.power_to_db(S, ref=np.max)

    n_bands, n_frames = S.shape

    return {
        "n_mels": int(n_bands),
        "n_frames": int(n_frames),
        "raw_samples": n,
        "mel_cells": int(n_bands * n_frames),
        "compression_ratio": round(n / (n_bands * n_frames), 4),
        "frames_per_second": round(sample_rate / hop_length, 4),
        "loudest_band": int(np.argmax(S_db.mean(axis=1))),
    }

In [14]:
sample_rate = 16000
duration_s = 3.0
tone_hz = 440.0
n_mels = 80
n_fft = 400
hop_length = 160

mel_spectrogram_report(sample_rate, duration_s, tone_hz, n_mels, n_fft, hop_length)

{'n_mels': 80,
 'n_frames': 301,
 'raw_samples': 48000,
 'mel_cells': 24080,
 'compression_ratio': 1.9934,
 'frames_per_second': 100.0,
 'loudest_band': 11}

In [15]:
sample_rate = 8000
duration_s = 2.0
tone_hz = 300.0
n_mels = 32
n_fft = 256
hop_length = 128

mel_spectrogram_report(sample_rate, duration_s, tone_hz, n_mels, n_fft, hop_length)

{'n_mels': 32,
 'n_frames': 126,
 'raw_samples': 16000,
 'mel_cells': 4032,
 'compression_ratio': 3.9683,
 'frames_per_second': 62.5,
 'loudest_band': 3}

### [Class 5] Question 6. Scoring a Transcript and Placing It on the Quality Scale

**Context:** A transcript is not judged by whether it looks right. The standard measure aligns it
against the reference word by word and counts the three ways the two can disagree — a wrong word, an
extra word, a missing word — then divides the total by the length of the reference. The rate alone is
not the whole story: the breakdown says *how* a system is failing, and a rate only becomes actionable
once it is placed against the bands the field uses, which is what separates "ship it" from "needs
post-processing".

**Problem Statement:**

Using the `jiwer` library, write a function `wer_quality_report(reference, hypothesis)`.

1. Align the hypothesis against the reference at word level and count the **substitutions**, the
   **deletions** and the **insertions**.
2. Compute the word error rate — the three counts summed and divided by the number of words in the
   reference — and round it to 4 decimal places. Report also the reference's word count, taking
   words to be whitespace-separated tokens.
3. Report the error total the rate implies for this reference: the rate multiplied by the reference
   word count, rounded to the nearest whole number.
4. Place the transcript on the quality scale by its rate expressed as a percentage: below 5 is
   `"excellent"`, from 5 up to but not including 15 is `"good"`, from 15 up to but not including 30
   is `"fair"`, and 30 or above is `"poor"`. Report the label verbatim, in lower case.
5. Return a dictionary with exactly these seven keys: `"substitutions"`, `"deletions"`,
   `"insertions"`, `"reference_words"` and `"errors"` as integers, `"wer"` as a float, and
   `"quality"` as one of the four strings above.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: reference (string)
    Single line: a string
  Argument 2: hypothesis (string)
    Single line: a string
```

**Output Format:**

For each test case, print the result of `wer_quality_report()` on a new line.

**Sample Input:**

```
reference  = "the flight departs at nine thirty from terminal two"
hypothesis = "the light departs at nine thirty from terminal to"
```

**Sample Output:**

```
{'substitutions': 2, 'deletions': 0, 'insertions': 0, 'wer': 0.2222, 'reference_words': 9, 'errors': 2, 'quality': 'fair'}
```

**Note:** The input handling is already in place — you only need to implement the function. Each
string arrives on a single line and is already lower-case with no punctuation, so no normalisation is
required. Reach for the library's alignment output rather than its rate alone: the same call that
scores a pair also reports how the errors break down. Round the rate before you report it, but derive
the error total from the unrounded rate.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: wer_quality_report
RETURN TYPE: Dictionary

ARGUMENTS:
1. reference  → String
2. hypothesis → String
```

---

### 3. Reference Solution Code

```python
import jiwer

QUALITY_BANDS = [(5.0, "excellent"), (15.0, "good"), (30.0, "fair")]


def wer_quality_report(reference: str, hypothesis: str) -> dict:
    out = jiwer.process_words(reference, hypothesis)
    reference_words = len(reference.split())
    rate = float(out.wer)

    quality = next((name for edge, name in QUALITY_BANDS if rate * 100 < edge), "poor")

    return {
        "substitutions": int(out.substitutions),
        "deletions": int(out.deletions),
        "insertions": int(out.insertions),
        "wer": round(rate, 4),
        "reference_words": reference_words,
        "errors": round(rate * reference_words),
        "quality": quality,
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
the flight departs at nine thirty from terminal two
the light departs at nine thirty from terminal to
```

**out1.txt**
```
{'substitutions': 2, 'deletions': 0, 'insertions': 0, 'wer': 0.2222, 'reference_words': 9, 'errors': 2, 'quality': 'fair'}
```

**in2.txt**
```
1
the quick brown fox jumps over the lazy dog
the quick brown fox jumps over the lazy dog
```

**out2.txt**
```
{'substitutions': 0, 'deletions': 0, 'insertions': 0, 'wer': 0.0, 'reference_words': 9, 'errors': 0, 'quality': 'excellent'}
```

**in3.txt**
```
1
please turn off the lights
please turn off lights
```

**out3.txt**
```
{'substitutions': 0, 'deletions': 1, 'insertions': 0, 'wer': 0.2, 'reference_words': 5, 'errors': 1, 'quality': 'fair'}
```

In [16]:
import jiwer

QUALITY_BANDS = [(5.0, "excellent"), (15.0, "good"), (30.0, "fair")]


def wer_quality_report(reference: str, hypothesis: str) -> dict:
    out = jiwer.process_words(reference, hypothesis)
    reference_words = len(reference.split())
    rate = float(out.wer)

    quality = next((name for edge, name in QUALITY_BANDS if rate * 100 < edge), "poor")

    return {
        "substitutions": int(out.substitutions),
        "deletions": int(out.deletions),
        "insertions": int(out.insertions),
        "wer": round(rate, 4),
        "reference_words": reference_words,
        "errors": round(rate * reference_words),
        "quality": quality,
    }

In [17]:
reference = 'the flight departs at nine thirty from terminal two'
hypothesis = 'the light departs at nine thirty from terminal to'

wer_quality_report(reference, hypothesis)

{'substitutions': 2,
 'deletions': 0,
 'insertions': 0,
 'wer': 0.2222,
 'reference_words': 9,
 'errors': 2,
 'quality': 'fair'}

In [18]:
reference = 'the quick brown fox jumps over the lazy dog'
hypothesis = 'the quick brown fox jumps over the lazy dog'

wer_quality_report(reference, hypothesis)

{'substitutions': 0,
 'deletions': 0,
 'insertions': 0,
 'wer': 0.0,
 'reference_words': 9,
 'errors': 0,
 'quality': 'excellent'}

In [19]:
reference = 'please turn off the lights'
hypothesis = 'please turn off lights'

wer_quality_report(reference, hypothesis)

{'substitutions': 0,
 'deletions': 1,
 'insertions': 0,
 'wer': 0.2,
 'reference_words': 5,
 'errors': 1,
 'quality': 'fair'}

### [Class 5] Question 7. Tubelet Tokens, the VideoMAE Mask, and What the Clip Costs

**Context:** A video transformer does not consume frames; it consumes tokens. The clip is cut into
small three-dimensional cubes — a few frames deep, a few pixels across — and each cube is flattened
into one token. That gives a token grid with a temporal axis as well as two spatial ones, and the
token count is what decides whether the clip fits: attention cost grows with its square. Self-
supervised pretraining then exploits the redundancy directly, hiding the overwhelming majority of
those tokens and asking the model to reconstruct them from the few that remain. Sizing all of this
before a run is a matter of dividing the clip up correctly and counting.

**Problem Statement:**

Write a function
`tubelet_budget(num_frames, height, width, channels, tubelet_frames, patch_size, keep_ratio, bytes_per_value)`.

1. The clip has shape `(num_frames, height, width, channels)`. Cut it into tubelets that span
   `tubelet_frames` frames and `patch_size × patch_size` pixels. Report the token grid as three
   separate counts — how many tubelets fit along time, along height, and along width — taking only
   whole tubelets along each axis and discarding any partial remainder.
2. Report the total number of tokens the grid holds, and the length of one token's flattened vector —
   every value inside one tubelet, across all channels.
3. Masked pretraining keeps only a fraction `keep_ratio` of the tokens visible. The number kept is
   that fraction of the token total, truncated to a whole number, but never fewer than one. Report
   how many tokens stay visible, how many are masked, and the masked share as a percentage of the
   total rounded to 2 decimal places.
4. Report what the raw clip costs in memory: every value occupies `bytes_per_value` bytes, and the
   total is expressed in megabytes of 10^6 bytes each, rounded to 2 decimal places.
5. Return a dictionary with exactly these nine keys: `"grid_t"`, `"grid_h"`, `"grid_w"`,
   `"num_tokens"`, `"token_dim"`, `"visible_tokens"` and `"masked_tokens"` as integers, and
   `"masked_pct"` and `"clip_mb"` as floats.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: num_frames (integer)
    Single line: one integer
  Argument 2: height (integer)
    Single line: one integer
  Argument 3: width (integer)
    Single line: one integer
  Argument 4: channels (integer)
    Single line: one integer
  Argument 5: tubelet_frames (integer)
    Single line: one integer
  Argument 6: patch_size (integer)
    Single line: one integer
  Argument 7: keep_ratio (float)
    Single line: one float
  Argument 8: bytes_per_value (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `tubelet_budget()` on a new line.

**Sample Input:**

```
num_frames      = 8
height          = 56
width           = 56
channels        = 3
tubelet_frames  = 2
patch_size      = 14
keep_ratio      = 0.10
bytes_per_value = 4
```

**Sample Output:**

```
{'grid_t': 4, 'grid_h': 4, 'grid_w': 4, 'num_tokens': 64, 'token_dim': 1176, 'visible_tokens': 6, 'masked_tokens': 58, 'masked_pct': 90.62, 'clip_mb': 0.3}
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is loaded and no clip is built: every figure follows from the eight numbers. The visible-token count
is truncated rather than rounded, so a ratio of one tenth on 64 tokens keeps 6 and not 7 — and the
masked share that follows is therefore not exactly the complement of `keep_ratio`.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: tubelet_budget
RETURN TYPE: Dictionary

ARGUMENTS:
1. num_frames      → Integer
2. height          → Integer
3. width           → Integer
4. channels        → Integer
5. tubelet_frames  → Integer
6. patch_size      → Integer
7. keep_ratio      → Float
8. bytes_per_value → Integer
```

---

### 3. Reference Solution Code

```python
def tubelet_budget(num_frames: int, height: int, width: int, channels: int,
                   tubelet_frames: int, patch_size: int, keep_ratio: float,
                   bytes_per_value: int) -> dict:
    grid_t = num_frames // tubelet_frames
    grid_h = height // patch_size
    grid_w = width // patch_size

    num_tokens = grid_t * grid_h * grid_w
    token_dim = tubelet_frames * patch_size * patch_size * channels

    visible = max(1, int(num_tokens * keep_ratio))
    masked = num_tokens - visible

    clip_bytes = num_frames * height * width * channels * bytes_per_value

    return {
        "grid_t": grid_t,
        "grid_h": grid_h,
        "grid_w": grid_w,
        "num_tokens": num_tokens,
        "token_dim": token_dim,
        "visible_tokens": visible,
        "masked_tokens": masked,
        "masked_pct": round(100.0 * masked / num_tokens, 2),
        "clip_mb": round(clip_bytes / 1e6, 2),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
8
56
56
3
2
14
0.10
4
```

**out1.txt**
```
{'grid_t': 4, 'grid_h': 4, 'grid_w': 4, 'num_tokens': 64, 'token_dim': 1176, 'visible_tokens': 6, 'masked_tokens': 58, 'masked_pct': 90.62, 'clip_mb': 0.3}
```

**in2.txt**
```
1
16
224
224
3
2
16
0.25
4
```

**out2.txt**
```
{'grid_t': 8, 'grid_h': 14, 'grid_w': 14, 'num_tokens': 1568, 'token_dim': 1536, 'visible_tokens': 392, 'masked_tokens': 1176, 'masked_pct': 75.0, 'clip_mb': 9.63}
```

**in3.txt**
```
1
300
224
224
3
4
16
0.10
4
```

**out3.txt**
```
{'grid_t': 75, 'grid_h': 14, 'grid_w': 14, 'num_tokens': 14700, 'token_dim': 3072, 'visible_tokens': 1470, 'masked_tokens': 13230, 'masked_pct': 90.0, 'clip_mb': 180.63}
```

In [20]:
def tubelet_budget(num_frames: int, height: int, width: int, channels: int,
                   tubelet_frames: int, patch_size: int, keep_ratio: float,
                   bytes_per_value: int) -> dict:
    grid_t = num_frames // tubelet_frames
    grid_h = height // patch_size
    grid_w = width // patch_size

    num_tokens = grid_t * grid_h * grid_w
    token_dim = tubelet_frames * patch_size * patch_size * channels

    visible = max(1, int(num_tokens * keep_ratio))
    masked = num_tokens - visible

    clip_bytes = num_frames * height * width * channels * bytes_per_value

    return {
        "grid_t": grid_t,
        "grid_h": grid_h,
        "grid_w": grid_w,
        "num_tokens": num_tokens,
        "token_dim": token_dim,
        "visible_tokens": visible,
        "masked_tokens": masked,
        "masked_pct": round(100.0 * masked / num_tokens, 2),
        "clip_mb": round(clip_bytes / 1e6, 2),
    }

In [21]:
num_frames = 8
height = 56
width = 56
channels = 3
tubelet_frames = 2
patch_size = 14
keep_ratio = 0.10
bytes_per_value = 4

tubelet_budget(num_frames, height, width, channels, tubelet_frames, patch_size, keep_ratio, bytes_per_value)

{'grid_t': 4,
 'grid_h': 4,
 'grid_w': 4,
 'num_tokens': 64,
 'token_dim': 1176,
 'visible_tokens': 6,
 'masked_tokens': 58,
 'masked_pct': 90.62,
 'clip_mb': 0.3}

In [22]:
num_frames = 16
height = 224
width = 224
channels = 3
tubelet_frames = 2
patch_size = 16
keep_ratio = 0.25
bytes_per_value = 4

tubelet_budget(num_frames, height, width, channels, tubelet_frames, patch_size, keep_ratio, bytes_per_value)

{'grid_t': 8,
 'grid_h': 14,
 'grid_w': 14,
 'num_tokens': 1568,
 'token_dim': 1536,
 'visible_tokens': 392,
 'masked_tokens': 1176,
 'masked_pct': 75.0,
 'clip_mb': 9.63}

In [23]:
num_frames = 300
height = 224
width = 224
channels = 3
tubelet_frames = 4
patch_size = 16
keep_ratio = 0.10
bytes_per_value = 4

tubelet_budget(num_frames, height, width, channels, tubelet_frames, patch_size, keep_ratio, bytes_per_value)

{'grid_t': 75,
 'grid_h': 14,
 'grid_w': 14,
 'num_tokens': 14700,
 'token_dim': 3072,
 'visible_tokens': 1470,
 'masked_tokens': 13230,
 'masked_pct': 90.0,
 'clip_mb': 180.63}

### [Class 6] Question 8. Synchronising Gradients Across Data-Parallel Replicas

**Context:** Data parallelism is the default way to use more than one GPU. Every device holds a
complete copy of the model, the batch is split between them, and each device runs its own forward and
backward pass — which means each device ends up with a *different* gradient, computed from different
samples. What makes the scheme correct rather than merely parallel is the collective that follows:
the per-device gradients are averaged so every replica applies the identical update. The claim worth
verifying is that this is not an approximation — the averaged gradient is exactly the gradient the
whole batch would have produced on one device.

**Problem Statement:**

Using `torch`, write a function `ddp_gradient_sync(features, targets, weights, world_size)`.

1. `features` is the global batch, one row per sample; `targets` holds one value per sample; and
   `weights` is the single set of starting parameters every replica begins from. Split the batch into
   `world_size` equal contiguous shards — the first shard is the first rows, and so on — and report
   the number of samples in one shard.
2. On each replica independently, predict each of its own samples as the dot product of its feature
   row with the weights, and take that replica's loss to be the **mean over its own shard** of the
   squared difference between prediction and target. Obtain the gradient of that loss with respect to
   the weights by automatic differentiation, starting from the same `weights` every time.
3. Report the L2 norm of each replica's gradient, in rank order, rounded to 4 decimal places — these
   differ, because each replica saw different samples.
4. Synchronise: report the element-wise **mean** of the per-replica gradients, rounded to 4 decimal
   places.
5. Separately, compute the gradient of the same loss taken as the mean over the **whole** batch on a
   single device, and report it rounded the same way. Report also whether the two agree to within
   `1e-4` on every element.
6. Return a dictionary with exactly these five keys: `"shard_size"` as an integer,
   `"rank_grad_norms"`, `"averaged_grad"` and `"single_device_grad"` as lists of floats, and
   `"equivalent"` as a boolean.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: features (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 2: targets (numpy_1d)
    Line 1: N (array length)
    Line 2: N space-separated values
  Argument 3: weights (numpy_1d)
    Line 1: N (array length)
    Line 2: N space-separated values
  Argument 4: world_size (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `ddp_gradient_sync()` on a new line.

**Sample Input:**

```
features = tensor([[1., 2.],
                   [2., 1.],
                   [3., 0.],
                   [0., 3.],
                   [1., 1.],
                   [2., 2.]])
targets    = array([3., 4., 5., 1., 2., 6.])
weights    = array([0.5, -0.25])
world_size = 3
```

**Sample Output:**

```
{'shard_size': 2, 'rank_grad_norms': [13.2594, 11.7394, 18.0312], 'averaged_grad': [-10.9167, -9.0833], 'single_device_grad': [-10.9167, -9.0833], 'equivalent': True}
```

**Note:** The input handling is already in place — you only need to implement the function. No process
group is created and no GPU is involved: the replicas are simulated one after another on one device,
which is the only way to compare them against the single-device answer. The batch size is always
divisible by `world_size`. Each replica must start from the supplied weights rather than from
whatever the previous replica left behind, so build a fresh differentiable parameter per replica —
gradients accumulate otherwise.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: ddp_gradient_sync
RETURN TYPE: Dictionary

ARGUMENTS:
1. features   → Torch 2D Tensor
2. targets    → Numpy 1D Array
3. weights    → Numpy 1D Array
4. world_size → Integer
```

---

### 3. Reference Solution Code

```python
import numpy as np
import torch


def ddp_gradient_sync(features: torch.Tensor, targets: np.ndarray,
                      weights: np.ndarray, world_size: int) -> dict:
    y = torch.as_tensor(targets, dtype=torch.float32)
    shard = features.shape[0] // world_size

    def grad_of(X, t):
        w = torch.tensor(weights, dtype=torch.float32, requires_grad=True)
        loss = ((X @ w - t) ** 2).mean()
        loss.backward()
        return w.grad.detach().clone()

    rank_grads = [grad_of(features[r * shard:(r + 1) * shard],
                          y[r * shard:(r + 1) * shard])
                  for r in range(world_size)]

    averaged = torch.stack(rank_grads).mean(dim=0)
    single = grad_of(features, y)

    return {
        "shard_size": shard,
        "rank_grad_norms": [round(float(torch.linalg.vector_norm(g)), 4) for g in rank_grads],
        "averaged_grad": [round(float(v), 4) for v in averaged],
        "single_device_grad": [round(float(v), 4) for v in single],
        "equivalent": bool(float(torch.max(torch.abs(averaged - single))) < 1e-4),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
6 2
1.0 2.0
2.0 1.0
3.0 0.0
0.0 3.0
1.0 1.0
2.0 2.0
6
3.0 4.0 5.0 1.0 2.0 6.0
2
0.5 -0.25
3
```

**out1.txt**
```
{'shard_size': 2, 'rank_grad_norms': [13.2594, 11.7394, 18.0312], 'averaged_grad': [-10.9167, -9.0833], 'single_device_grad': [-10.9167, -9.0833], 'equivalent': True}
```

**in2.txt**
```
1
8 3
1.0 0.0 2.0
0.0 1.0 1.0
2.0 2.0 0.0
1.0 3.0 1.0
3.0 1.0 2.0
0.0 2.0 3.0
2.0 0.0 1.0
1.0 1.0 1.0
8
2.0 1.0 4.0 3.0 5.0 2.0 3.0 1.0
3
0.1 0.2 -0.3
4
```

**out2.txt**
```
{'shard_size': 2, 'rank_grad_norms': [6.6836, 17.5579, 25.4831, 8.3457], 'averaged_grad': [-8.6, -6.7, -7.625], 'single_device_grad': [-8.6, -6.7, -7.625], 'equivalent': True}
```

In [24]:
import numpy as np
import torch


def ddp_gradient_sync(features: torch.Tensor, targets: np.ndarray,
                      weights: np.ndarray, world_size: int) -> dict:
    y = torch.as_tensor(targets, dtype=torch.float32)
    shard = features.shape[0] // world_size

    def grad_of(X, t):
        w = torch.tensor(weights, dtype=torch.float32, requires_grad=True)
        loss = ((X @ w - t) ** 2).mean()
        loss.backward()
        return w.grad.detach().clone()

    rank_grads = [grad_of(features[r * shard:(r + 1) * shard],
                          y[r * shard:(r + 1) * shard])
                  for r in range(world_size)]

    averaged = torch.stack(rank_grads).mean(dim=0)
    single = grad_of(features, y)

    return {
        "shard_size": shard,
        "rank_grad_norms": [round(float(torch.linalg.vector_norm(g)), 4) for g in rank_grads],
        "averaged_grad": [round(float(v), 4) for v in averaged],
        "single_device_grad": [round(float(v), 4) for v in single],
        "equivalent": bool(float(torch.max(torch.abs(averaged - single))) < 1e-4),
    }

In [25]:
features = torch.tensor([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 0.0],
    [0.0, 3.0],
    [1.0, 1.0],
    [2.0, 2.0]
], dtype=torch.float32)
targets = np.array([3.0, 4.0, 5.0, 1.0, 2.0, 6.0])
weights = np.array([0.5, -0.25])
world_size = 3

ddp_gradient_sync(features, targets, weights, world_size)

{'shard_size': 2,
 'rank_grad_norms': [13.2594, 11.7394, 18.0312],
 'averaged_grad': [-10.9167, -9.0833],
 'single_device_grad': [-10.9167, -9.0833],
 'equivalent': True}

In [26]:
features = torch.tensor([
    [1.0, 0.0, 2.0],
    [0.0, 1.0, 1.0],
    [2.0, 2.0, 0.0],
    [1.0, 3.0, 1.0],
    [3.0, 1.0, 2.0],
    [0.0, 2.0, 3.0],
    [2.0, 0.0, 1.0],
    [1.0, 1.0, 1.0]
], dtype=torch.float32)
targets = np.array([2.0, 1.0, 4.0, 3.0, 5.0, 2.0, 3.0, 1.0])
weights = np.array([0.1, 0.2, -0.3])
world_size = 4

ddp_gradient_sync(features, targets, weights, world_size)

{'shard_size': 2,
 'rank_grad_norms': [6.6836, 17.5579, 25.4831, 8.3457],
 'averaged_grad': [-8.6, -6.7, -7.625],
 'single_device_grad': [-8.6, -6.7, -7.625],
 'equivalent': True}

### [Class 6] Question 9. Splitting an MLP Block Column-Then-Row Across GPUs

**Context:** Tensor parallelism splits work *inside* a single matrix multiplication. A weight matrix
can be cut along its output dimension, so each device produces a slice of the output; or along its
input dimension, so each device produces a partial sum of the whole output. Which cut you choose
decides which collective you need afterwards, and the two cuts compose: if the first matrix of a pair
is cut one way and the second the other way, each device can carry its own slice straight through
from one matrix into the next and the pair costs a single collective at the end instead of one after
each. That pairing is why tensor parallelism is affordable inside a transformer block at all.

**Problem Statement:**

Using `torch`, write a function `tensor_parallel_mlp(x, w1, w2, tp_degree)`.

1. The block computes, on one device, a first projection `x @ w1`, then a **ReLU**, then a second
   projection into the output: `relu(x @ w1) @ w2`. Compute that unsharded result; it is the answer
   the sharded version must reproduce.
2. Now shard across `tp_degree` devices. `w1` is split along its **output** dimension into
   `tp_degree` equal contiguous column blocks, and `w2` along its **input** dimension into the
   matching row blocks — device `s` gets column block `s` of `w1` and row block `s` of `w2`. Report
   how many hidden units one device owns, and the shape of each of its two weight shards as
   `[rows, columns]`.
3. Each device computes its own share of the block end to end: the full input against its `w1` shard,
   the activation applied to that, and the result against its `w2` shard. What each device holds is a
   partial sum of the whole output, not a slice of it.
4. Combine the devices' partials into the final output with the collective that partial sums require,
   and report how many collectives the whole block needed.
5. Report the sum of every element of device 0's partial, and the sum of every element of the combined
   output, each rounded to 4 decimal places; and whether the combined output agrees with step 1's
   unsharded result to within `1e-4` on every element.
6. Return a dictionary with exactly these seven keys: `"hidden_per_gpu"` and
   `"collectives_per_block"` as integers, `"w1_shard_shape"` and `"w2_shard_shape"` as two-element
   lists, `"gpu0_partial_sum"` and `"output_sum"` as floats, and `"matches"` as a boolean.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: x (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 2: w1 (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 3: w2 (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 4: tp_degree (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `tensor_parallel_mlp()` on a new line.

**Sample Input:**

```
x  = tensor([[ 1., -2.,  0.5],
             [ 0.5, 1., -1. ]])
w1 = tensor([[ 0.5, -1.,  0.25, 2. ],
             [ 1.,   0.5, -0.5, 0. ],
             [-2.,   1.,   1.5, 0.5]])
w2 = tensor([[ 1.,   0.5],
             [-0.5,  2. ],
             [ 0.25,-1. ],
             [ 2.,   1.5]])
tp_degree = 2
```

**Sample Output:**

```
{'hidden_per_gpu': 2, 'w1_shard_shape': [3, 2], 'w2_shard_shape': [2, 2], 'collectives_per_block': 1, 'gpu0_partial_sum': 4.875, 'output_sum': 13.0, 'matches': True}
```

**Note:** The input handling is already in place — you only need to implement the function. No
process group is created and no GPU is involved: every device's share is computed in turn on one
device, which is what lets you compare the result against the unsharded one. The hidden dimension is
always divisible by `tp_degree`. Every device sees the **whole** input `x` — it is the weights that
are split, not the batch. The activation is applied on each device to that device's own hidden slice.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: tensor_parallel_mlp
RETURN TYPE: Dictionary

ARGUMENTS:
1. x         → Torch 2D Tensor
2. w1        → Torch 2D Tensor
3. w2        → Torch 2D Tensor
4. tp_degree → Integer
```

---

### 3. Reference Solution Code

```python
import torch


def tensor_parallel_mlp(x: torch.Tensor, w1: torch.Tensor, w2: torch.Tensor,
                        tp_degree: int) -> dict:
    hidden = w1.shape[1]
    per_gpu = hidden // tp_degree

    y_full = torch.relu(x @ w1) @ w2

    partials = []
    for s in range(tp_degree):
        w1_shard = w1[:, s * per_gpu:(s + 1) * per_gpu]   # column-parallel
        w2_shard = w2[s * per_gpu:(s + 1) * per_gpu, :]   # row-parallel
        partials.append(torch.relu(x @ w1_shard) @ w2_shard)

    y_tp = torch.stack(partials).sum(dim=0)               # all-reduce (sum)

    return {
        "hidden_per_gpu": per_gpu,
        "w1_shard_shape": [int(w1.shape[0]), per_gpu],
        "w2_shard_shape": [per_gpu, int(w2.shape[1])],
        "collectives_per_block": 1,
        "gpu0_partial_sum": round(float(partials[0].sum()), 4),
        "output_sum": round(float(y_tp.sum()), 4),
        "matches": bool(float(torch.max(torch.abs(y_tp - y_full))) < 1e-4),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
2 3
1.0 -2.0 0.5
0.5 1.0 -1.0
3 4
0.5 -1.0 0.25 2.0
1.0 0.5 -0.5 0.0
-2.0 1.0 1.5 0.5
4 2
1.0 0.5
-0.5 2.0
0.25 -1.0
2.0 1.5
2
```

**out1.txt**
```
{'hidden_per_gpu': 2, 'w1_shard_shape': [3, 2], 'w2_shard_shape': [2, 2], 'collectives_per_block': 1, 'gpu0_partial_sum': 4.875, 'output_sum': 13.0, 'matches': True}
```

**in2.txt**
```
1
3 2
2.0 1.0
0.0 -1.0
1.0 3.0
2 6
1.0 -0.5 2.0 0.5 -1.0 1.5
0.5 2.0 -1.0 1.0 0.25 -0.5
6 2
1.0 0.0
0.5 -1.0
-2.0 1.5
1.0 1.0
0.25 2.0
-0.5 0.5
3
```

**out2.txt**
```
{'hidden_per_gpu': 2, 'w1_shard_shape': [2, 2], 'w2_shard_shape': [2, 2], 'collectives_per_block': 1, 'gpu0_partial_sum': 1.75, 'output_sum': 10.75, 'matches': True}
```

In [27]:
import torch


def tensor_parallel_mlp(x: torch.Tensor, w1: torch.Tensor, w2: torch.Tensor,
                        tp_degree: int) -> dict:
    hidden = w1.shape[1]
    per_gpu = hidden // tp_degree

    y_full = torch.relu(x @ w1) @ w2

    partials = []
    for s in range(tp_degree):
        w1_shard = w1[:, s * per_gpu:(s + 1) * per_gpu]   # column-parallel
        w2_shard = w2[s * per_gpu:(s + 1) * per_gpu, :]   # row-parallel
        partials.append(torch.relu(x @ w1_shard) @ w2_shard)

    y_tp = torch.stack(partials).sum(dim=0)               # all-reduce (sum)

    return {
        "hidden_per_gpu": per_gpu,
        "w1_shard_shape": [int(w1.shape[0]), per_gpu],
        "w2_shard_shape": [per_gpu, int(w2.shape[1])],
        "collectives_per_block": 1,
        "gpu0_partial_sum": round(float(partials[0].sum()), 4),
        "output_sum": round(float(y_tp.sum()), 4),
        "matches": bool(float(torch.max(torch.abs(y_tp - y_full))) < 1e-4),
    }

In [28]:
x = torch.tensor([
    [1.0, -2.0, 0.5],
    [0.5, 1.0, -1.0]
], dtype=torch.float32)
w1 = torch.tensor([
    [0.5, -1.0, 0.25, 2.0],
    [1.0, 0.5, -0.5, 0.0],
    [-2.0, 1.0, 1.5, 0.5]
], dtype=torch.float32)
w2 = torch.tensor([
    [1.0, 0.5],
    [-0.5, 2.0],
    [0.25, -1.0],
    [2.0, 1.5]
], dtype=torch.float32)
tp_degree = 2

tensor_parallel_mlp(x, w1, w2, tp_degree)

{'hidden_per_gpu': 2,
 'w1_shard_shape': [3, 2],
 'w2_shard_shape': [2, 2],
 'collectives_per_block': 1,
 'gpu0_partial_sum': 4.875,
 'output_sum': 13.0,
 'matches': True}

In [29]:
x = torch.tensor([
    [2.0, 1.0],
    [0.0, -1.0],
    [1.0, 3.0]
], dtype=torch.float32)
w1 = torch.tensor([
    [1.0, -0.5, 2.0, 0.5, -1.0, 1.5],
    [0.5, 2.0, -1.0, 1.0, 0.25, -0.5]
], dtype=torch.float32)
w2 = torch.tensor([
    [1.0, 0.0],
    [0.5, -1.0],
    [-2.0, 1.5],
    [1.0, 1.0],
    [0.25, 2.0],
    [-0.5, 0.5]
], dtype=torch.float32)
tp_degree = 3

tensor_parallel_mlp(x, w1, w2, tp_degree)

{'hidden_per_gpu': 2,
 'w1_shard_shape': [2, 2],
 'w2_shard_shape': [2, 2],
 'collectives_per_block': 1,
 'gpu0_partial_sum': 1.75,
 'output_sum': 10.75,
 'matches': True}

### [Class 7] Question 10. Per-Group Quantization Against a Single Tensor-Wide Scale

**Context:** Granularity is the first dial you turn when INT8 weights cost you accuracy. One scale
for an entire weight matrix is the cheapest possible mapping — a single number of metadata, no
per-row bookkeeping — but it is set by the largest magnitude anywhere in the matrix, so one outlier
squeezes every ordinary weight into a handful of integer codes. Cutting the weights into
fixed-size groups and giving each group its own scale confines that damage to the group holding the
outlier, at the cost of storing one extra float per group. Whether that trade is worth making is an
empirical question: quantize both ways, dequantize both, and compare the reconstruction error and
the metadata each way demands.

**Problem Statement:**

Using the `torch` library, write a function `per_group_quantization_report(weights, group_size)`.

1. Both schemes below use **symmetric (absmax)** INT8 quantization: for a given set of values the
   scale is that set's largest absolute value divided by the largest positive INT8 code, and no
   zero-point/offset is used. Signed INT8 spans `[-128, 127]`. A code is rounded to the nearest
   integer and then held inside that range.
2. **Per-group:** read `weights` as a flat sequence of its elements in row-major order and cut that
   sequence into consecutive groups of exactly `group_size` elements. Every group gets its own
   independently computed scale. `group_size` always divides the number of columns.
3. Dequantize the per-group codes using the group scale that produced each of them, and compute the
   mean squared error against `weights`, averaged over all elements.
4. **Per-tensor:** quantize and dequantize the same tensor again using one scale computed from the
   whole tensor, and compute its mean squared error the same way.
5. Return a dictionary with exactly five keys:
   - `"num_groups"` — how many groups the tensor was cut into
   - `"num_scales"` — how many scale values the per-group scheme has to store alongside its codes
   - `"mse_per_group"` — the per-group reconstruction error, rounded to 6 decimal places
   - `"mse_per_tensor"` — the per-tensor reconstruction error, rounded to 6 decimal places
   - `"error_reduction_pct"` — the percentage of the per-tensor error that the per-group scheme
     removes, rounded to 2 decimal places

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: weights (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 2: group_size (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `per_group_quantization_report()` on a new line.

**Sample Input:**

```
weights = [[ 0.82, -1.34,  0.05,  2.10, -0.47,  0.93],
           [88.50, -0.62,  0.31, -1.75,  0.08,  1.42],
           [-0.24,  0.66, -2.98,  0.51, -0.13,  0.77],
           [ 1.05, -0.39,  0.22, -0.94,  3.61, -1.28]]
group_size = 3
```

**Sample Output:**

```
{'num_groups': 8, 'num_scales': 8, 'mse_per_group': 0.004257, 'mse_per_tensor': 0.035554, 'error_reduction_pct': 88.03}
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is loaded or trained. `weights` arrives as a `float32` tensor; leave it in that dtype. Do the
quantize/dequantize arithmetic at full precision and round only the numbers you return — rounding a
scale early can shift a code by a whole step and move the error you report. Note that groups are
cut from the flattened element sequence, so a group is not the same thing as a row unless
`group_size` happens to equal the row width.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: per_group_quantization_report
RETURN TYPE: Dictionary

ARGUMENTS:
1. weights    → Torch 2D Tensor
2. group_size → Integer
```

---

### 3. Reference Solution Code

```python
import torch


def per_group_quantization_report(weights: torch.Tensor, group_size: int) -> dict:
    q_min, q_max = torch.iinfo(torch.int8).min, torch.iinfo(torch.int8).max
    original_shape = weights.shape

    def absmax_scale(t):
        return t.abs().max().item() / q_max

    def quantize(t, scale):
        return torch.round(t / scale).clamp(q_min, q_max).to(torch.int8)

    # ---- per-group ----
    grouped = weights.view(-1, group_size)
    num_groups = grouped.shape[0]
    group_scales = torch.zeros(num_groups)
    for i in range(num_groups):
        group_scales[i] = absmax_scale(grouped.select(0, i))
    group_scales = group_scales.view(-1, 1)

    q_group = quantize(grouped, group_scales)
    deq_group = (group_scales * q_group.float()).view(original_shape)
    mse_group = (weights - deq_group).square().mean().item()

    # ---- per-tensor ----
    tensor_scale = absmax_scale(weights)
    q_tensor = quantize(weights, tensor_scale)
    deq_tensor = tensor_scale * q_tensor.float()
    mse_tensor = (weights - deq_tensor).square().mean().item()

    return {
        "num_groups": int(num_groups),
        "num_scales": int(group_scales.numel()),
        "mse_per_group": round(mse_group, 6),
        "mse_per_tensor": round(mse_tensor, 6),
        "error_reduction_pct": round((1 - mse_group / mse_tensor) * 100, 2),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
4 6
0.82 -1.34 0.05 2.10 -0.47 0.93
88.50 -0.62 0.31 -1.75 0.08 1.42
-0.24 0.66 -2.98 0.51 -0.13 0.77
1.05 -0.39 0.22 -0.94 3.61 -1.28
3
```

**out1.txt**
```
{'num_groups': 8, 'num_scales': 8, 'mse_per_group': 0.004257, 'mse_per_tensor': 0.035554, 'error_reduction_pct': 88.03}
```

**in2.txt**
```
1
2 8
12.40 -0.85 0.44 -3.10 0.09 0.71 -0.28 1.63
-0.52 0.37 -46.20 0.95 2.08 -1.11 0.64 -0.19
4
```

**out2.txt**
```
{'num_groups': 4, 'num_scales': 4, 'mse_per_group': 0.003015, 'mse_per_tensor': 0.011954, 'error_reduction_pct': 74.78}
```

In [38]:
import torch


def per_group_quantization_report(weights: torch.Tensor, group_size: int) -> dict:
    q_min, q_max = torch.iinfo(torch.int8).min, torch.iinfo(torch.int8).max
    original_shape = weights.shape

    def absmax_scale(t):
        return t.abs().max().item() / q_max

    def quantize(t, scale):
        return torch.round(t / scale).clamp(q_min, q_max).to(torch.int8)

    # ---- per-group ----
    grouped = weights.view(-1, group_size)
    num_groups = grouped.shape[0]
    group_scales = torch.zeros(num_groups)
    for i in range(num_groups):
        group_scales[i] = absmax_scale(grouped.select(0, i))
    group_scales = group_scales.view(-1, 1)

    q_group = quantize(grouped, group_scales)
    deq_group = (group_scales * q_group.float()).view(original_shape)
    mse_group = (weights - deq_group).square().mean().item()

    # ---- per-tensor ----
    tensor_scale = absmax_scale(weights)
    q_tensor = quantize(weights, tensor_scale)
    deq_tensor = tensor_scale * q_tensor.float()
    mse_tensor = (weights - deq_tensor).square().mean().item()

    return {
        "num_groups": int(num_groups),
        "num_scales": int(group_scales.numel()),
        "mse_per_group": round(mse_group, 6),
        "mse_per_tensor": round(mse_tensor, 6),
        "error_reduction_pct": round((1 - mse_group / mse_tensor) * 100, 2),
    }

In [39]:
weights = torch.tensor([
    [0.82, -1.34, 0.05, 2.10, -0.47, 0.93],
    [88.50, -0.62, 0.31, -1.75, 0.08, 1.42],
    [-0.24, 0.66, -2.98, 0.51, -0.13, 0.77],
    [1.05, -0.39, 0.22, -0.94, 3.61, -1.28]
], dtype=torch.float32)
group_size = 3

per_group_quantization_report(weights, group_size)

{'num_groups': 8,
 'num_scales': 8,
 'mse_per_group': 0.004257,
 'mse_per_tensor': 0.035554,
 'error_reduction_pct': 88.03}

In [40]:
weights = torch.tensor([
    [12.40, -0.85, 0.44, -3.10, 0.09, 0.71, -0.28, 1.63],
    [-0.52, 0.37, -46.20, 0.95, 2.08, -1.11, 0.64, -0.19]
], dtype=torch.float32)
group_size = 4

per_group_quantization_report(weights, group_size)

{'num_groups': 4,
 'num_scales': 4,
 'mse_per_group': 0.003015,
 'mse_per_tensor': 0.011954,
 'error_reduction_pct': 74.78}

### [Class 7] Question 11. Static Calibration — Observed Activations to Frozen Quantization Parameters

**Context:** Weights are static, so their quantization parameters can be computed the moment you
have the checkpoint. Activations cannot: they depend on the input, and no scale exists until you
have seen some data. The standard answer is a calibration pass — push a small, representative
sample through the model, record the range each activation tensor actually reaches, and convert
those ranges into a scale and a zero-point that are then **frozen** for every future inference.
The risk is baked in at that moment. Anything the calibration sample never reached will be clamped
at the ends of the integer range once real traffic arrives, so the number worth reporting alongside
the parameters is how much of that traffic the frozen range failed to cover.

**Problem Statement:**

Using the `torch` library, write a function
`calibrate_activation_quantizers(layer_names, calibration_acts, production_acts)`.

1. Row `i` of `calibration_acts` holds every activation value the layer named `layer_names[i]`
   produced during the calibration pass. Row `i` of `production_acts` holds the values that same
   layer produced later, on live input. Both tensors have one row per layer, but they need not have
   the same number of columns.
2. For each layer, take the minimum and the maximum observed over its **calibration** row and turn
   that range into **asymmetric (zero-point)** INT8 quantization parameters: a floating-point
   scale, and an integer zero-point that must itself be a valid INT8 code. Signed INT8 spans
   `[-128, 127]`.
3. Freeze those parameters and quantize the layer's **production** row with them: scale and shift
   each value, round to the nearest integer code, then hold the result inside the signed INT8 range.
4. A production value whose code had to be pulled back to an end of the INT8 range could not be
   represented by the frozen parameters. Report what fraction of the layer's production values that
   was.
5. Return a dictionary mapping each layer name to a dictionary with exactly three keys:
   - `"scale"` — that layer's scale, rounded to 6 decimal places
   - `"zero_point"` — that layer's zero-point, as an integer
   - `"clipped_fraction"` — the fraction of that layer's production values whose code had to be
     pulled back into range, rounded to 4 decimal places

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: layer_names (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: calibration_acts (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 3: production_acts (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
```

**Output Format:**

For each test case, print the result of `calibrate_activation_quantizers()` on a new line.

**Sample Input:**

```
layer_names = ["layer_0", "layer_1", "layer_2", "layer_3"]
calibration_acts = [[  -0.460,   0.498,  0.120, -0.310,   0.205, -0.077,   0.361, -0.152],
                    [   0.840,  -0.668,  0.214, -0.395,   0.502, -0.121,   0.733, -0.288],
                    [-140.672,  21.588, -3.415,  0.926, -18.204,  7.331, -55.470, 12.083],
                    [  21.602, -140.717, -2.884,  1.475, -21.660,  6.902, -49.318, 10.554]]
production_acts  = [[  0.310, -0.402,   0.495,  0.640, -0.512,  0.055, -0.221,  0.5024,  0.5005, -0.088],
                    [  0.512, -0.334,   0.905,  1.204, -0.590,  0.087, -0.701,  0.446,   0.760,  -0.155],
                    [-95.400, 15.820, -142.500,  3.640, -12.075, 22.980, -55.310, 8.902, -160.750, 1.204],
                    [-88.200, 19.440,  -3.115,  0.550, -21.330,  6.780, -49.100, 10.220, -1.902,  12.640]]
```

**Sample Output:**

```
{'layer_0': {'scale': 0.003757, 'zero_point': -6, 'clipped_fraction': 0.3}, 'layer_1': {'scale': 0.005914, 'zero_point': -15, 'clipped_fraction': 0.3}, 'layer_2': {'scale': 0.636314, 'zero_point': 93, 'clipped_fraction': 0.3}, 'layer_3': {'scale': 0.636545, 'zero_point': 93, 'clipped_fraction': 0.0}}
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is loaded or trained; both tensors arrive as `float32` and should be left in that dtype. The
statistics are gathered per layer, so a layer's parameters depend only on its own row, never on the
tensor as a whole.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: calibrate_activation_quantizers
RETURN TYPE: Dictionary

ARGUMENTS:
1. layer_names      → List of Strings
2. calibration_acts → Torch 2D Tensor
3. production_acts  → Torch 2D Tensor
```

---

### 3. Reference Solution Code

```python
import torch


def calibrate_activation_quantizers(layer_names: list, calibration_acts: torch.Tensor,
                                    production_acts: torch.Tensor) -> dict:
    q_min, q_max = torch.iinfo(torch.int8).min, torch.iinfo(torch.int8).max

    report = {}
    for i, name in enumerate(layer_names):
        observed = calibration_acts.select(0, i)
        r_min, r_max = observed.min().item(), observed.max().item()

        scale = (r_max - r_min) / (q_max - q_min)
        zero_point = int(round(q_min - r_min / scale))
        zero_point = max(q_min, min(q_max, zero_point))

        live = production_acts.select(0, i)
        codes = torch.round(live / scale + zero_point)
        clipped = ((codes < q_min) | (codes > q_max)).sum().item()

        report[name] = {
            "scale": round(scale, 6),
            "zero_point": zero_point,
            "clipped_fraction": round(clipped / live.numel(), 4),
        }

    return report
```

---

### 4. Test Cases

**in1.txt**
```
1
4
layer_0
layer_1
layer_2
layer_3
4 8
-0.460 0.498 0.120 -0.310 0.205 -0.077 0.361 -0.152
0.840 -0.668 0.214 -0.395 0.502 -0.121 0.733 -0.288
-140.672 21.588 -3.415 0.926 -18.204 7.331 -55.470 12.083
21.602 -140.717 -2.884 1.475 -21.660 6.902 -49.318 10.554
4 10
0.310 -0.402 0.495 0.640 -0.512 0.055 -0.221 0.5024 0.5005 -0.088
0.512 -0.334 0.905 1.204 -0.590 0.087 -0.701 0.446 0.760 -0.155
-95.400 15.820 -142.500 3.640 -12.075 22.980 -55.310 8.902 -160.750 1.204
-88.200 19.440 -3.115 0.550 -21.330 6.780 -49.100 10.220 -1.902 12.640
```

**out1.txt**
```
{'layer_0': {'scale': 0.003757, 'zero_point': -6, 'clipped_fraction': 0.3}, 'layer_1': {'scale': 0.005914, 'zero_point': -15, 'clipped_fraction': 0.3}, 'layer_2': {'scale': 0.636314, 'zero_point': 93, 'clipped_fraction': 0.3}, 'layer_3': {'scale': 0.636545, 'zero_point': 93, 'clipped_fraction': 0.0}}
```

**in2.txt**
```
1
3
block_0
block_1
block_2
3 6
0.050 6.450 1.204 3.318 0.877 5.012
-12.400 3.950 -0.884 1.207 -6.315 2.048
-2.480 2.510 0.005 -1.115 1.902 -0.337
3 5
0.120 6.380 7.200 2.455 0.049
-20.500 -14.200 8.900 -13.050 5.600
1.204 -2.310 0.008 2.480 -1.775
```

**out2.txt**
```
{'block_0': {'scale': 0.025098, 'zero_point': -128, 'clipped_fraction': 0.2}, 'block_1': {'scale': 0.064118, 'zero_point': 65, 'clipped_fraction': 1.0}, 'block_2': {'scale': 0.019569, 'zero_point': -1, 'clipped_fraction': 0.0}}
```

In [41]:
import torch


def calibrate_activation_quantizers(layer_names: list, calibration_acts: torch.Tensor,
                                    production_acts: torch.Tensor) -> dict:
    q_min, q_max = torch.iinfo(torch.int8).min, torch.iinfo(torch.int8).max

    report = {}
    for i, name in enumerate(layer_names):
        observed = calibration_acts.select(0, i)
        r_min, r_max = observed.min().item(), observed.max().item()

        scale = (r_max - r_min) / (q_max - q_min)
        zero_point = int(round(q_min - r_min / scale))
        zero_point = max(q_min, min(q_max, zero_point))

        live = production_acts.select(0, i)
        codes = torch.round(live / scale + zero_point)
        clipped = ((codes < q_min) | (codes > q_max)).sum().item()

        report[name] = {
            "scale": round(scale, 6),
            "zero_point": zero_point,
            "clipped_fraction": round(clipped / live.numel(), 4),
        }

    return report

In [42]:
layer_names = ['layer_0', 'layer_1', 'layer_2', 'layer_3']
calibration_acts = torch.tensor([
    [-0.460, 0.498, 0.120, -0.310, 0.205, -0.077, 0.361, -0.152],
    [0.840, -0.668, 0.214, -0.395, 0.502, -0.121, 0.733, -0.288],
    [-140.672, 21.588, -3.415, 0.926, -18.204, 7.331, -55.470, 12.083],
    [21.602, -140.717, -2.884, 1.475, -21.660, 6.902, -49.318, 10.554]
], dtype=torch.float32)
production_acts = torch.tensor([
    [0.310, -0.402, 0.495, 0.640, -0.512, 0.055, -0.221, 0.5024, 0.5005, -0.088],
    [0.512, -0.334, 0.905, 1.204, -0.590, 0.087, -0.701, 0.446, 0.760, -0.155],
    [-95.400, 15.820, -142.500, 3.640, -12.075, 22.980, -55.310, 8.902, -160.750, 1.204],
    [-88.200, 19.440, -3.115, 0.550, -21.330, 6.780, -49.100, 10.220, -1.902, 12.640]
], dtype=torch.float32)

calibrate_activation_quantizers(layer_names, calibration_acts, production_acts)

{'layer_0': {'scale': 0.003757, 'zero_point': -6, 'clipped_fraction': 0.3},
 'layer_1': {'scale': 0.005914, 'zero_point': -15, 'clipped_fraction': 0.3},
 'layer_2': {'scale': 0.636314, 'zero_point': 93, 'clipped_fraction': 0.3},
 'layer_3': {'scale': 0.636545, 'zero_point': 93, 'clipped_fraction': 0.0}}

In [43]:
layer_names = ['block_0', 'block_1', 'block_2']
calibration_acts = torch.tensor([
    [0.050, 6.450, 1.204, 3.318, 0.877, 5.012],
    [-12.400, 3.950, -0.884, 1.207, -6.315, 2.048],
    [-2.480, 2.510, 0.005, -1.115, 1.902, -0.337]
], dtype=torch.float32)
production_acts = torch.tensor([
    [0.120, 6.380, 7.200, 2.455, 0.049],
    [-20.500, -14.200, 8.900, -13.050, 5.600],
    [1.204, -2.310, 0.008, 2.480, -1.775]
], dtype=torch.float32)

calibrate_activation_quantizers(layer_names, calibration_acts, production_acts)

{'block_0': {'scale': 0.025098, 'zero_point': -128, 'clipped_fraction': 0.2},
 'block_1': {'scale': 0.064118, 'zero_point': 65, 'clipped_fraction': 1.0},
 'block_2': {'scale': 0.019569, 'zero_point': -1, 'clipped_fraction': 0.0}}

### [Class 7] Question 12. Ternary Weights and What 1.58 Bits Buys

**Context:** The most aggressive weight formats give up on integers-with-a-scale altogether and let
each weight take one of three values: -1, 0, or +1. Three levels per weight is under two bits, and
because the levels are exactly those three, a matrix multiply degenerates into additions and skips
— the zeros drop terms entirely, and nothing needs a multiplier. A ternary weight matrix is
produced by scaling the tensor by its mean absolute value and rounding onto the three levels, and
the interesting question is what the result costs: how the weights distribute across the three
levels, how much fidelity the reconstruction gives up, and how large the whole model becomes at
that width.

**Problem Statement:**

Using the `torch` library, write a function
`ternary_weight_report(weights, total_params, baseline_bits)`.

1. Quantize `weights` onto the ternary levels `{-1, 0, 1}` using **absmean** quantization: the
   scale is the mean absolute value of the whole tensor. Divide by that scale, round to the nearest
   integer, and hold the result within the three levels.
2. Count how many weights landed on each of the three levels.
3. Dequantize — the ternary levels rescaled by the same scale — and compute the mean squared error
   against `weights`, averaged over all elements.
4. A three-level code carries log₂3 bits per weight. At that width, compute the footprint of a
   whole model of `total_params` parameters in MB (1 MB = 1024 × 1024 bytes), and the compression
   ratio against the same model stored at `baseline_bits` bits per parameter.
5. Return a dictionary with exactly six keys:
   - `"scale"` — the absmean scale, rounded to 6 decimal places
   - `"counts"` — a dictionary mapping each of the three integer levels `-1`, `0` and `1` to how
     many weights landed on it
   - `"error_mse"` — the reconstruction error, rounded to 6 decimal places
   - `"bits_per_weight"` — the effective width in bits, rounded to 4 decimal places
   - `"footprint_mb"` — the ternary model's footprint in MB, rounded to 2 decimal places
   - `"compression_ratio"` — how many times smaller that is than the `baseline_bits` baseline,
     rounded to 2 decimal places

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: weights (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 2: total_params (integer)
    Single line: one integer
  Argument 3: baseline_bits (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `ternary_weight_report()` on a new line.

**Sample Input:**

```
weights = [[ 0.62, -1.35,  0.08,  2.10],
           [-0.41,  0.91, -0.05, -1.72],
           [ 1.28,  0.33, -0.79,  0.17],
           [-2.41,  0.56,  1.04, -0.28]]
total_params = 3000000000
baseline_bits = 16
```

**Sample Output:**

```
{'scale': 0.88125, 'counts': {-1: 4, 0: 6, 1: 6}, 'error_mse': 0.343979, 'bits_per_weight': 1.585, 'footprint_mb': 566.83, 'compression_ratio': 10.09}
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is loaded or trained. `weights` arrives as a `float32` tensor; leave it in that dtype.
`total_params` is the parameter count of the whole model being costed out, not of the tensor you
were handed — the tensor is there to be quantized, the count to be priced.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: ternary_weight_report
RETURN TYPE: Dictionary

ARGUMENTS:
1. weights       → Torch 2D Tensor
2. total_params  → Integer
3. baseline_bits → Integer
```

---

### 3. Reference Solution Code

```python
import math

import torch


def ternary_weight_report(weights: torch.Tensor, total_params: int, baseline_bits: int) -> dict:
    scale = weights.abs().mean().item()
    ternary = torch.round(weights / scale).clamp(-1, 1)

    counts = {level: int((ternary == level).sum().item()) for level in (-1, 0, 1)}

    dequantized = scale * ternary
    mse = (weights - dequantized).square().mean().item()

    bits_per_weight = math.log2(3)
    footprint_mb = total_params * bits_per_weight / 8 / (1024 * 1024)

    return {
        "scale": round(scale, 6),
        "counts": counts,
        "error_mse": round(mse, 6),
        "bits_per_weight": round(bits_per_weight, 4),
        "footprint_mb": round(footprint_mb, 2),
        "compression_ratio": round(baseline_bits / bits_per_weight, 2),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
4 4
0.62 -1.35 0.08 2.10
-0.41 0.91 -0.05 -1.72
1.28 0.33 -0.79 0.17
-2.41 0.56 1.04 -0.28
3000000000
16
```

**out1.txt**
```
{'scale': 0.88125, 'counts': {-1: 4, 0: 6, 1: 6}, 'error_mse': 0.343979, 'bits_per_weight': 1.585, 'footprint_mb': 566.83, 'compression_ratio': 10.09}
```

**in2.txt**
```
1
4 5
1.85 -0.12 0.44 -2.63 0.07
-0.98 3.21 -0.35 0.16 -1.47
0.52 -0.71 2.08 -0.09 0.63
-1.19 0.28 -0.86 1.34 -0.03
70000000000
32
```

**out2.txt**
```
{'scale': 0.9505, 'counts': {-1: 6, 0: 8, 1: 6}, 'error_mse': 0.564662, 'bits_per_weight': 1.585, 'footprint_mb': 13225.96, 'compression_ratio': 20.19}
```

In [44]:
import math

import torch


def ternary_weight_report(weights: torch.Tensor, total_params: int, baseline_bits: int) -> dict:
    scale = weights.abs().mean().item()
    ternary = torch.round(weights / scale).clamp(-1, 1)

    counts = {level: int((ternary == level).sum().item()) for level in (-1, 0, 1)}

    dequantized = scale * ternary
    mse = (weights - dequantized).square().mean().item()

    bits_per_weight = math.log2(3)
    footprint_mb = total_params * bits_per_weight / 8 / (1024 * 1024)

    return {
        "scale": round(scale, 6),
        "counts": counts,
        "error_mse": round(mse, 6),
        "bits_per_weight": round(bits_per_weight, 4),
        "footprint_mb": round(footprint_mb, 2),
        "compression_ratio": round(baseline_bits / bits_per_weight, 2),
    }

In [45]:
weights = torch.tensor([
    [0.62, -1.35, 0.08, 2.10],
    [-0.41, 0.91, -0.05, -1.72],
    [1.28, 0.33, -0.79, 0.17],
    [-2.41, 0.56, 1.04, -0.28]
], dtype=torch.float32)
total_params = 3000000000
baseline_bits = 16

ternary_weight_report(weights, total_params, baseline_bits)

{'scale': 0.88125,
 'counts': {-1: 4, 0: 6, 1: 6},
 'error_mse': 0.343979,
 'bits_per_weight': 1.585,
 'footprint_mb': 566.83,
 'compression_ratio': 10.09}

In [46]:
weights = torch.tensor([
    [1.85, -0.12, 0.44, -2.63, 0.07],
    [-0.98, 3.21, -0.35, 0.16, -1.47],
    [0.52, -0.71, 2.08, -0.09, 0.63],
    [-1.19, 0.28, -0.86, 1.34, -0.03]
], dtype=torch.float32)
total_params = 70000000000
baseline_bits = 32

ternary_weight_report(weights, total_params, baseline_bits)

{'scale': 0.9505,
 'counts': {-1: 6, 0: 8, 1: 6},
 'error_mse': 0.564662,
 'bits_per_weight': 1.585,
 'footprint_mb': 13225.96,
 'compression_ratio': 20.19}

### [Class 8] Question 13. Flattening and Sharding One FSDP Unit

**Context:** Sharding a model is not a matter of handing each device a different layer. Inside one
FSDP unit every parameter tensor — weights and biases alike, whatever their shapes — is first
flattened and concatenated into a single one-dimensional buffer, and it is *that* buffer which is
cut into equal pieces, one per device. Equal pieces means the buffer has to divide evenly, so it is
padded up first, and a device can easily end up holding a slice that straddles what used to be two
different tensors, or nothing but padding. Peak memory follows from the same picture: a device holds
its own slice at all times, plus one whole unit for as long as it is being computed on.

**Problem Statement:**

Using `torch`, write a function `fsdp_shard_unit(unit_weight, unit_bias, world_size)`.

1. The unit holds two parameters: `unit_weight`, a 2-D tensor, and `unit_bias`, a 1-D array. Flatten
   each to one dimension and concatenate them **in that order** into a single buffer. Report how
   many elements it holds.
2. The buffer is split into `world_size` equal shards, so it must first be padded with zeros up to
   the next multiple of `world_size`. Report how many padding elements were added — zero when it
   already divides evenly — and how many elements each shard holds.
3. Report the contents of the shard held by rank 0 and the shard held by the last rank, each as a
   list of values rounded to 4 decimal places.
4. While a unit is being computed on, its device holds its own shard **and** the whole unflattened
   unit gathered from every rank. Report that peak as a number of elements: one shard plus the
   unpadded buffer.
5. Return a dictionary with exactly these six keys: `"flat_numel"`, `"pad_elements"`,
   `"shard_numel"` and `"peak_numel"` as integers, and `"rank0_shard"` and `"last_shard"` as lists
   of floats.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: unit_weight (torch_tensor_2d)
    Line 1: R C (rows cols)
    Next R lines: C space-separated values each
  Argument 2: unit_bias (numpy_1d)
    Line 1: N (array length)
    Line 2: N space-separated values
  Argument 3: world_size (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `fsdp_shard_unit()` on a new line.

**Sample Input:**

```
unit_weight = tensor([[ 1.,  2.,  3.,  4.],
                      [ 5.,  6.,  7.,  8.],
                      [ 9., 10., 11., 12.]])
unit_bias   = array([13., 14., 15.])
world_size  = 16
```

**Sample Output:**

```
{'flat_numel': 15, 'pad_elements': 1, 'shard_numel': 1, 'rank0_shard': [1.0], 'last_shard': [0.0], 'peak_numel': 16}
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
is loaded and no distributed process group is started: this is the layout one unit would have, worked
out on a single device. The weight arrives as a `float32` tensor and the bias as a floating-point
array, so bring them into the same buffer before splitting. `world_size` may exceed the number of
elements in the unit, which is a real configuration and not an error.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: fsdp_shard_unit
RETURN TYPE: Dictionary

ARGUMENTS:
1. unit_weight → Torch 2D Tensor
2. unit_bias   → Numpy 1D Array
3. world_size  → Integer
```

---

### 3. Reference Solution Code

```python
import numpy as np
import torch


def fsdp_shard_unit(unit_weight: torch.Tensor, unit_bias: np.ndarray,
                    world_size: int) -> dict:
    flat = torch.cat([unit_weight.reshape(-1),
                      torch.as_tensor(unit_bias, dtype=torch.float32)])

    pad = (-flat.numel()) % world_size
    padded = torch.cat([flat, torch.zeros(pad)])
    shards = padded.chunk(world_size)

    return {
        "flat_numel": int(flat.numel()),
        "pad_elements": int(pad),
        "shard_numel": int(shards[0].numel()),
        "rank0_shard": [round(float(v), 4) for v in shards[0]],
        "last_shard": [round(float(v), 4) for v in shards[-1]],
        "peak_numel": int(shards[0].numel() + flat.numel()),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
3 4
1.0 2.0 3.0 4.0
5.0 6.0 7.0 8.0
9.0 10.0 11.0 12.0
3
13.0 14.0 15.0
16
```

**out1.txt**
```
{'flat_numel': 15, 'pad_elements': 1, 'shard_numel': 1, 'rank0_shard': [1.0], 'last_shard': [0.0], 'peak_numel': 16}
```

**in2.txt**
```
1
4 5
-1.0 -0.895 -0.789 -0.684 -0.579
-0.474 -0.368 -0.263 -0.158 -0.053
0.053 0.158 0.263 0.368 0.474
0.579 0.684 0.789 0.895 1.0
4
0.5 -0.25 0.125 2.0
4
```

**out2.txt**
```
{'flat_numel': 24, 'pad_elements': 0, 'shard_numel': 6, 'rank0_shard': [-1.0, -0.895, -0.789, -0.684, -0.579, -0.474], 'last_shard': [0.895, 1.0, 0.5, -0.25, 0.125, 2.0], 'peak_numel': 30}
```

**in3.txt**
```
1
2 3
1.5 -2.5 3.5
0.5 -0.5 4.5
2
9.0 -9.0
3
```

**out3.txt**
```
{'flat_numel': 8, 'pad_elements': 1, 'shard_numel': 3, 'rank0_shard': [1.5, -2.5, 3.5], 'last_shard': [9.0, -9.0, 0.0], 'peak_numel': 11}
```

In [47]:
import numpy as np
import torch


def fsdp_shard_unit(unit_weight: torch.Tensor, unit_bias: np.ndarray,
                    world_size: int) -> dict:
    flat = torch.cat([unit_weight.reshape(-1),
                      torch.as_tensor(unit_bias, dtype=torch.float32)])

    pad = (-flat.numel()) % world_size
    padded = torch.cat([flat, torch.zeros(pad)])
    shards = padded.chunk(world_size)

    return {
        "flat_numel": int(flat.numel()),
        "pad_elements": int(pad),
        "shard_numel": int(shards[0].numel()),
        "rank0_shard": [round(float(v), 4) for v in shards[0]],
        "last_shard": [round(float(v), 4) for v in shards[-1]],
        "peak_numel": int(shards[0].numel() + flat.numel()),
    }

In [48]:
unit_weight = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0, 12.0]
], dtype=torch.float32)
unit_bias = np.array([13.0, 14.0, 15.0])
world_size = 16

fsdp_shard_unit(unit_weight, unit_bias, world_size)

{'flat_numel': 15,
 'pad_elements': 1,
 'shard_numel': 1,
 'rank0_shard': [1.0],
 'last_shard': [0.0],
 'peak_numel': 16}

In [49]:
unit_weight = torch.tensor([
    [-1.0, -0.895, -0.789, -0.684, -0.579],
    [-0.474, -0.368, -0.263, -0.158, -0.053],
    [0.053, 0.158, 0.263, 0.368, 0.474],
    [0.579, 0.684, 0.789, 0.895, 1.0]
], dtype=torch.float32)
unit_bias = np.array([0.5, -0.25, 0.125, 2.0])
world_size = 4

fsdp_shard_unit(unit_weight, unit_bias, world_size)

{'flat_numel': 24,
 'pad_elements': 0,
 'shard_numel': 6,
 'rank0_shard': [-1.0, -0.895, -0.789, -0.684, -0.579, -0.474],
 'last_shard': [0.895, 1.0, 0.5, -0.25, 0.125, 2.0],
 'peak_numel': 30}

In [50]:
unit_weight = torch.tensor([
    [1.5, -2.5, 3.5],
    [0.5, -0.5, 4.5]
], dtype=torch.float32)
unit_bias = np.array([9.0, -9.0])
world_size = 3

fsdp_shard_unit(unit_weight, unit_bias, world_size)

{'flat_numel': 8,
 'pad_elements': 1,
 'shard_numel': 3,
 'rank0_shard': [1.5, -2.5, 3.5],
 'last_shard': [9.0, -9.0, 0.0],
 'peak_numel': 11}

### [Class 9] Question 14. The Cost of Not Caching

**Context:** An autoregressive decoder emits one token at a time, and each step attends over every
token before it. The naive implementation re-runs the whole forward pass over the whole sequence at
every step, which recomputes key and value vectors that were already computed one step earlier —
those vectors never change once a token is fixed. Caching them collapses the total projection work
over a run from quadratic in the sequence length to linear, and the size of that gap is the number
you quote when someone asks why the cache is worth the memory it eats.

**Problem Statement:**

Write a function `kv_cache_recompute_saving(prompt_len, num_new_tokens, num_layers)`.

1. A run over a prompt of `prompt_len` tokens that emits `num_new_tokens` tokens consists of
   `num_new_tokens + 1` forward passes: one pass over the prompt, then one further pass for each
   token emitted.
2. Whenever a forward pass processes a token position, it computes one key vector and one value
   vector for that position at each of the model's `num_layers` layers.
3. A pass sees every position of the sequence as it stands when the pass runs: the first pass sees
   `prompt_len` positions, and the pass that runs once `i` tokens have already been emitted sees
   `prompt_len + i` positions.
4. **With a cache**, a position's key and value vectors are computed only by the first pass that
   sees that position; every later pass reads them back instead of recomputing them. Count the key
   and value vectors computed over the whole run in this mode.
5. **Without a cache**, every pass recomputes the key and value vectors for every position it sees.
   Count the key and value vectors computed over the whole run in this mode.
6. Return a dictionary with keys `"with_cache"` (integer), `"without_cache"` (integer) and
   `"ratio"` (float) — the last being `without_cache / with_cache` rounded to 4 decimal places.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: prompt_len (integer)
    Single line: one integer
  Argument 2: num_new_tokens (integer)
    Single line: one integer
  Argument 3: num_layers (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `kv_cache_recompute_saving()` on a new line.

**Sample Input:**

```
prompt_len = 9
num_new_tokens = 50
num_layers = 12
```

**Sample Output:**

```
{'with_cache': 1416, 'without_cache': 41616, 'ratio': 29.3898}
```

**Note:** The input handling is already in place — you only need to implement the function. This is
arithmetic on a generation schedule; nothing is loaded and no tensor work is involved. Keep both
counts as exact integers — they grow fast, which is the whole point of the comparison.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: kv_cache_recompute_saving
RETURN TYPE: Dictionary

ARGUMENTS:
1. prompt_len      → Integer
2. num_new_tokens  → Integer
3. num_layers      → Integer
```

---

### 3. Reference Solution Code

```python
def kv_cache_recompute_saving(prompt_len: int, num_new_tokens: int, num_layers: int) -> dict:
    with_cache_positions = prompt_len + num_new_tokens
    without_cache_positions = sum(prompt_len + i for i in range(num_new_tokens + 1))

    with_cache = 2 * num_layers * with_cache_positions
    without_cache = 2 * num_layers * without_cache_positions

    return {
        "with_cache": with_cache,
        "without_cache": without_cache,
        "ratio": round(without_cache / with_cache, 4),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
9
50
12
```

**out1.txt**
```
{'with_cache': 1416, 'without_cache': 41616, 'ratio': 29.3898}
```

**in2.txt**
```
1
1024
256
32
```

**out2.txt**
```
{'with_cache': 81920, 'without_cache': 18948096, 'ratio': 231.3}
```

In [51]:
def kv_cache_recompute_saving(prompt_len: int, num_new_tokens: int, num_layers: int) -> dict:
    with_cache_positions = prompt_len + num_new_tokens
    without_cache_positions = sum(prompt_len + i for i in range(num_new_tokens + 1))

    with_cache = 2 * num_layers * with_cache_positions
    without_cache = 2 * num_layers * without_cache_positions

    return {
        "with_cache": with_cache,
        "without_cache": without_cache,
        "ratio": round(without_cache / with_cache, 4),
    }

In [52]:
prompt_len = 9
num_new_tokens = 50
num_layers = 12

kv_cache_recompute_saving(prompt_len, num_new_tokens, num_layers)

{'with_cache': 1416, 'without_cache': 41616, 'ratio': 29.3898}

In [53]:
prompt_len = 1024
num_new_tokens = 256
num_layers = 32

kv_cache_recompute_saving(prompt_len, num_new_tokens, num_layers)

{'with_cache': 81920, 'without_cache': 18948096, 'ratio': 231.3}

### [Class 9] Question 15. Measuring What the KV Cache Actually Costs

**Context:** The reason a decoder can emit a token without re-reading its whole context is that the
key and value vectors for everything already seen are kept in a per-request buffer. That buffer is
the price: it grows by a fixed amount for every token, at every layer, and on a long conversation it
is what runs the GPU out of memory rather than the weights. The per-token cost is predictable from
the model's shape — layer count, key/value head count, head width, and the size of the dtype — but a
prediction is not a measurement. The buffer is a real object you can inspect after a forward pass,
and reading its size directly is how you find out whether your arithmetic matched your model.

**Problem Statement:**

Using the `transformers` and `torch` libraries, write a function
`kv_cache_measured_growth(model_id, prompt_ids, num_new_tokens)`.

1. Load the causal language model `model_id` in 32-bit floating point and put it in evaluation mode.
2. Run one forward pass over the whole of `prompt_ids` with caching enabled — the prefill. Keep the
   cache it returns.
3. Then take `num_new_tokens` decode steps. Each step feeds **only the single most recently produced
   token**, together with the cache from the step before, and takes the next token to be the highest
   scoring entry of the last position's output scores.
4. From the cache as it stands at the end, report: how many layers it holds; the number of key/value
   heads and the width of one head, both read off a stored key tensor; and how many token positions
   are cached.
5. Report the cache's size in bytes twice — as it stood right after the prefill, and as it stands at
   the end. Size a cache by summing, over every layer, the bytes occupied by its stored keys and its
   stored values. Report also the final size divided by the number of cached positions, rounded to 4
   decimal places.
6. Return a dictionary with exactly these seven keys: `"num_layers"`, `"kv_heads"`, `"head_dim"`,
   `"cached_tokens"`, `"bytes_after_prefill"` and `"bytes_after_decode"` as integers, and
   `"bytes_per_token"` as a float.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: model_id (string)
    Single line: a string
  Argument 2: prompt_ids (list_int)
    Line 1: N (list length)
    Line 2: N space-separated integers
  Argument 3: num_new_tokens (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `kv_cache_measured_growth()` on a new line.

**Sample Input:**

```
model_id = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
prompt_ids = [11, 5, 2, 9, 14]
num_new_tokens = 3
```

**Sample Output:**

```
{'num_layers': 2, 'kv_heads': 2, 'head_dim': 2, 'cached_tokens': 8, 'bytes_after_prefill': 320, 'bytes_after_decode': 512, 'bytes_per_token': 64.0}
```

**Note:** The input handling is already in place — you only need to implement the function. The
models are tiny test checkpoints, so loading one inside the function is cheap and no GPU is
involved. Run the passes under no-gradient mode: gradients are never needed here and they cost
memory. A stored key tensor carries four axes — batch, heads, positions and head width — and which
is which is worth confirming on the prefill before you trust it after eight decode steps. Feeding
the whole sequence again at each step instead of the newest token would produce the same text and
defeat the entire point of the exercise.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: kv_cache_measured_growth
RETURN TYPE: Dictionary

ARGUMENTS:
1. model_id       → String
2. prompt_ids     → List of Ints
3. num_new_tokens → Integer
```

---

### 3. Reference Solution Code

```python
import torch
from transformers import AutoModelForCausalLM


def kv_cache_measured_growth(model_id: str, prompt_ids: list,
                             num_new_tokens: int) -> dict:
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
    model.eval()

    def cache_bytes(cache):
        return sum(layer.keys.numel() * layer.keys.element_size()
                   + layer.values.numel() * layer.values.element_size()
                   for layer in cache.layers)

    with torch.no_grad():
        out = model(torch.tensor([prompt_ids]), use_cache=True)
        after_prefill = cache_bytes(out.past_key_values)

        for _ in range(num_new_tokens):
            next_id = int(out.logits[0, -1].argmax())
            out = model(torch.tensor([[next_id]]),
                        past_key_values=out.past_key_values, use_cache=True)

    cache = out.past_key_values
    keys = cache.layers[0].keys
    after_decode = cache_bytes(cache)
    cached_tokens = int(keys.shape[2])

    return {
        "num_layers": len(cache.layers),
        "kv_heads": int(keys.shape[1]),
        "head_dim": int(keys.shape[3]),
        "cached_tokens": cached_tokens,
        "bytes_after_prefill": after_prefill,
        "bytes_after_decode": after_decode,
        "bytes_per_token": round(after_decode / cached_tokens, 4),
    }
```

---

### 4. Test Cases

**in1.txt**
```
1
trl-internal-testing/tiny-Qwen2ForCausalLM-2.5
5
11 5 2 9 14
3
```

**out1.txt**
```
{'num_layers': 2, 'kv_heads': 2, 'head_dim': 2, 'cached_tokens': 8, 'bytes_after_prefill': 320, 'bytes_after_decode': 512, 'bytes_per_token': 64.0}
```

**in2.txt**
```
1
trl-internal-testing/small-Qwen2ForCausalLM-2.5
2
4 7
2
```

**out2.txt**
```
{'num_layers': 2, 'kv_heads': 2, 'head_dim': 32, 'cached_tokens': 4, 'bytes_after_prefill': 2048, 'bytes_after_decode': 4096, 'bytes_per_token': 1024.0}
```

**in3.txt**
```
1
trl-internal-testing/tiny-Qwen2ForCausalLM-2.5
9
1 2 3 4 5 6 7 8 9
1
```

**out3.txt**
```
{'num_layers': 2, 'kv_heads': 2, 'head_dim': 2, 'cached_tokens': 10, 'bytes_after_prefill': 576, 'bytes_after_decode': 640, 'bytes_per_token': 64.0}
```

In [54]:
import torch
from transformers import AutoModelForCausalLM


def kv_cache_measured_growth(model_id: str, prompt_ids: list,
                             num_new_tokens: int) -> dict:
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
    model.eval()

    def cache_bytes(cache):
        return sum(layer.keys.numel() * layer.keys.element_size()
                   + layer.values.numel() * layer.values.element_size()
                   for layer in cache.layers)

    with torch.no_grad():
        out = model(torch.tensor([prompt_ids]), use_cache=True)
        after_prefill = cache_bytes(out.past_key_values)

        for _ in range(num_new_tokens):
            next_id = int(out.logits[0, -1].argmax())
            out = model(torch.tensor([[next_id]]),
                        past_key_values=out.past_key_values, use_cache=True)

    cache = out.past_key_values
    keys = cache.layers[0].keys
    after_decode = cache_bytes(cache)
    cached_tokens = int(keys.shape[2])

    return {
        "num_layers": len(cache.layers),
        "kv_heads": int(keys.shape[1]),
        "head_dim": int(keys.shape[3]),
        "cached_tokens": cached_tokens,
        "bytes_after_prefill": after_prefill,
        "bytes_after_decode": after_decode,
        "bytes_per_token": round(after_decode / cached_tokens, 4),
    }

In [55]:
model_id = 'trl-internal-testing/tiny-Qwen2ForCausalLM-2.5'
prompt_ids = [11, 5, 2, 9, 14]
num_new_tokens = 3

kv_cache_measured_growth(model_id, prompt_ids, num_new_tokens)


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 9450.57it/s]

{'num_layers': 2,
 'kv_heads': 2,
 'head_dim': 2,
 'cached_tokens': 8,
 'bytes_after_prefill': 320,
 'bytes_after_decode': 512,
 'bytes_per_token': 64.0}

In [56]:
model_id = 'trl-internal-testing/small-Qwen2ForCausalLM-2.5'
prompt_ids = [4, 7]
num_new_tokens = 2

kv_cache_measured_growth(model_id, prompt_ids, num_new_tokens)


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 8634.20it/s]

{'num_layers': 2,
 'kv_heads': 2,
 'head_dim': 32,
 'cached_tokens': 4,
 'bytes_after_prefill': 2048,
 'bytes_after_decode': 4096,
 'bytes_per_token': 1024.0}

In [57]:
model_id = 'trl-internal-testing/tiny-Qwen2ForCausalLM-2.5'
prompt_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9]
num_new_tokens = 1

kv_cache_measured_growth(model_id, prompt_ids, num_new_tokens)


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 27/27 [00:00<00:00, 9519.69it/s]

{'num_layers': 2,
 'kv_heads': 2,
 'head_dim': 2,
 'cached_tokens': 10,
 'bytes_after_prefill': 576,
 'bytes_after_decode': 640,
 'bytes_per_token': 64.0}